# DACON 236749: 성능 저하 진단과 AST 비교 실험

기존 225개 외부 검증 파일은 모두 개발 자료로 취급한다. 이미 결과를 확인한 65개 holdout으로 다시 모델을 선택하고 이를 독립 검증이라고 부르지 않는다. 새로운 음성 원문·음악 아티스트·악기 단위로 개발 자료와 두 평가 집합을 나눈다. 실제 실행 수치와 선택 결과는 `RESULTS_V2.md` 및 Drive 실행 폴더에 별도로 기록한다.

```mermaid
flowchart TD
    A[기존 G0–G4 예측과 정답] --> B[ROC 교차점·오분류 수·AUC·순위 변화 진단]
    C[MLAAD 실제·합성 음성] --> F[원본·라이선스·해시 보존]
    D[Echoes TTA와 기준 곡의 FMA 실제 음악] --> F
    E[NSynth 실제 악기 녹음] --> F
    F --> G[중복 제거·출처 연결 관계별 분리]
    G --> H[dev_train]
    G --> I[dev_select]
    G --> J[봉인된 test_generator / test_source]
    H --> K[AST 고정 + 다섯 분류 헤드 학습]
    K --> L[AST 마지막 블록 + 정규화 + 헤드 미세조정]
    M[G0: PANNs → Demucs → DF-Arena] --> N[개발 집합 비교]
    K --> N
    L --> N
    I --> N
    B --> N
    N --> O[출처 연결 집단 bootstrap·지표별 악화 제한]
    O --> P[모델·설정·개발 manifest 해시 잠금]
    P --> Q[새 평가 집합에서 세 모델을 한 번 비교]
    J --> Q
    Q --> R[독립 실행 ZIP·예측 일치·60초 오디오 부하 확인]
    R --> S[노트북·원본·가중치·로그·결과 Drive 저장 및 GPU 해제]
```

## 기존 결과에서 확인된 문제

| 비교 | 기존 holdout에서 확인한 변화 | 해석 범위 |
|---|---|---|
| G1 음성 | EER 0.041958 → 0.083916, AUC 0.980769 → 0.984266 | 순위 전체가 나빠진 것은 아니다. 공식 EER 교차점의 FP/FN이 각각 1개에서 2개로 늘었다. |
| G3 음악 | EER 0.307895 → 0.384211, AUC 0.721053 유지 | 개선된 순위 쌍과 악화된 순위 쌍이 상쇄돼도 EER 교차점은 바뀐다. |
| G4 파일 | 개발 EER 0.187500 → 0.143229, holdout 0.123476 → 0.156504 | 개발 집합의 이득이 이전 holdout으로 이어지지 않았다. 출처 편향이 유일한 원인이라고 단정할 수는 없다. |

이전 holdout 65개는 원자 출처 21개, 혼합 파일의 부모를 연결하면 독립적으로 재표집할 수 있는 연결 집단은 17개다. 클립 수를 그대로 표본 수로 보는 bootstrap은 불확실성을 과소평가할 수 있다. 이전 자료에는 음성 부재 정답도 없어 음성 존재 AUC를 계산할 수 없었다.

## 비교할 모델과 근거

| 후보 | 학습 범위 | 검증할 가설 |
|---|---|---|
| G0 | 기존 가중치와 계산 유지 | 현재 기준선의 성능을 동일한 새 자료에서 측정한다. |
| AST_FROZEN | AudioSet AST 표현 고정, 5개 출력 헤드 학습 | 음원 분리 없이 전체 음향 표현만으로 성분 존재와 생성 여부를 구분할 수 있는지 본다. |
| AST_LAST_BLOCK | 위 헤드에서 시작해 마지막 encoder 블록·최종 LayerNorm·헤드 학습 | 표현 일부를 실제·합성 구분에 맞추면 고정 표현보다 나아지는지 본다. |

AST는 AudioSet에서 학습된 약 86.6M 파라미터 모델이다. 작은 학습 집합에서 전체 모델을 갱신하면 과적합과 비용이 커질 수 있어 마지막 블록부터 비교한다. 고정 표현 실험과 미세조정 실험은 같은 데이터·전처리·출력 구조를 사용한다. 학습되는 파라미터의 gradient와 가중치 변화도 기록해, 분류기 학습과 backbone 미세조정을 구분한다. [모델과 라이선스](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593/tree/f826b80d28226b62986cc218e5cec390b1096902)

입력은 16 kHz 모노로 변환한다. 추론에서는 10초 이하 구간을 순서대로 나누고 마지막 구간까지 처리한 뒤 확률을 평균한다. 한 파일의 출력에 다른 파일의 예측이나 통계를 사용하지 않는다. 평균 집계는 짧은 합성 구간을 희석할 수 있으므로, 이번 비교의 고정된 한계로 기록한다.

정답이 없는 성분은 loss와 지표에서 제외한다. `VOICE_FAKE`와 `MUSIC_FAKE`는 해당 성분의 존재 정답이 1일 때만 학습한다. 음악의 제목·장르·프롬프트로 보컬 부재를 추정하지 않는다. NSynth의 실제 비음성 악기 녹음은 음성 부재 대조군이며 일반 음악 전체를 대표하지 않는다.

## 모델 선택과 예산

모델과 epoch 선택에는 `dev_select`만 사용한다. 계산 가능한 공식 지표의 가중치를 다시 정규화한 값은 **부분 정답 선택용 proxy**이며 DACON 점수가 아니다. G0 대비 proxy 개선 0.005 이상, 연결 집단 bootstrap 승률 0.70 이상, 개선량 10백분위수 −0.005 이상을 요구한다. 개별 EER 악화는 0.02, 대응 AUC 악화는 0.01 이내여야 한다. 기준을 통과하지 못하면 G0를 유지한다.

선택을 확정하고 모델·설정·개발 자료 해시를 저장한 뒤 새 평가 집합을 연다. 새 평가 결과로 후보를 재선택하거나 하이퍼파라미터를 바꾸지 않는다. 생성기와 원문이 학습에 없다는 것은 이번 실험의 데이터 분리를 뜻한다. 외부 사전학습 모델이 해당 데이터 계열을 전혀 본 적 없다는 뜻은 아니다.

데이터 다운로드와 CPU 검증, Drive 업로드를 끝낸 뒤 A100을 연결한다. 할당 시간은 최대 2시간이며 마지막 20분은 패키징과 저장에 남긴다. 오류가 나면 로그를 보존하고 런타임을 해제한다. A100에서 측정한 시간·메모리는 L4 서버의 60분 통과 증거로 해석하지 않는다.


## 1. Drive 연결
데이터와 모델 업로드를 마친 뒤 실행합니다. 최대 할당 시간은 2시간이며, 실행 오류와 정상 종료 모두 로그를 저장한 뒤 GPU를 해제합니다.

In [1]:
import time, json, shutil, subprocess, sys, zipfile, hashlib, threading, traceback
from pathlib import Path
from google.colab import drive, runtime
SESSION_START = time.time()
HARD_DEADLINE = SESSION_START + 7080
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
WORK = Path('/content/dacon236749_v2')
WORK.mkdir(exist_ok=True)
RUN = DRIVE / 'runs_v2' / time.strftime('%Y%m%d_%H%M%S', time.gmtime(SESSION_START))
RUN.mkdir(parents=True, exist_ok=True)
(RUN / 'budget.json').write_text(json.dumps({'start_unix':SESSION_START, 'hard_deadline':HARD_DEADLINE,
    'max_seconds_after_first_cell':7080, 'allocation_limit_seconds':7200}))
def budget_stop():
    try:
        (RUN / 'budget_stop.json').write_text(json.dumps({'unix':time.time(), 'reason':'allocation_limit'}))
    finally:
        runtime.unassign()
watchdog = threading.Timer(max(1,HARD_DEADLINE-time.time()), budget_stop)
watchdog.daemon = True
watchdog.start()
print('RESULTS_DIRECTORY', RUN, flush=True)

Mounted at /content/drive
RESULTS_DIRECTORY /content/drive/MyDrive/DACON_236749/runs_v2/20260912_174330


## 2. 실제 실행 코드
이 셀들의 내용을 실행 폴더와 Drive에 저장합니다. 데이터 분리·학습·선택·평가·패키징을 코드에서 확인할 수 있습니다.

In [2]:
%%writefile /content/dacon236749_v2/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


Writing /content/dacon236749_v2/graph_features.py


In [3]:
%%writefile /content/dacon236749_v2/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


Writing /content/dacon236749_v2/graph_fusion.py


In [4]:
%%writefile /content/dacon236749_v2/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


Writing /content/dacon236749_v2/graph_scoring.py


In [5]:
%%writefile /content/dacon236749_v2/run_experiment.py
"""A bounded pilot; every GPU result is checkpointed to the mounted Drive."""

import argparse
import csv
import importlib.metadata
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from graph_features import Extractor, atomic_json, extract_cached
from graph_fusion import fit_fusion, graph_predictions
from graph_scoring import official_metrics, validate_predictions


def read_csv(path: Path) -> list:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list) -> None:
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def reference_parity(extractor: Extractor, path: Path, feature) -> dict:
    """Compare the shared graph with untouched official functions on real audio."""
    base = extractor.base
    audio = base.load_audio(path)
    vp, mp = base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
    extractor.demucs.cpu()
    voice, music = base.separate_voice_and_music(path, extractor.demucs, extractor.device)
    extractor.demucs.to(extractor.device)
    vf = base.predict_fake(extractor.df, extractor.fake_index, voice, extractor.device)
    mf = base.predict_fake(extractor.df, extractor.fake_index, music, extractor.device)
    expected = np.array([base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp])
    actual = np.array(list(graph_predictions(feature)['G0'].values())[1:])
    delta = float(np.max(np.abs(expected - actual)))
    if not np.allclose(expected, actual, rtol=1e-5, atol=1e-6):
        raise AssertionError(f'G0 differs from official inference by {delta}: {path}')
    return {'ID': path.stem, 'max_probability_delta': delta, 'reference': expected.tolist()}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--baseline', type=Path, required=True)
    parser.add_argument('--validation', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--deadline', type=float, required=True)
    args = parser.parse_args()
    out = args.output
    out.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if start >= args.deadline:
        raise TimeoutError('The existing GPU budget has expired; no new run was started')
    labels = read_csv(args.validation / 'manifest.csv')
    if len({r['ID'] for r in labels}) != len(labels):
        raise ValueError('Duplicate validation IDs')
    paths = {r['ID']: args.validation / r['path'] for r in labels}
    if any(p.stem != key or not p.is_file() for key, p in paths.items()):
        raise ValueError('Manifest ID/path mismatch or missing audio')
    atomic_json(out / 'environment.json', {
        'python': sys.version, 'gpu': torch.cuda.get_device_name(),
        'gpu_total_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'started_unix': start, 'deadline_unix': args.deadline,
        'packages': {k: importlib.metadata.version(k) for k in
            ('torch', 'torchaudio', 'transformers', 'numpy', 'scipy', 'scikit-learn', 'demucs', 'librosa')},
    })
    extractor = Extractor(args.baseline / 'script.py')
    try:
        dummy = sorted((args.baseline / 'data' / 'test').glob('*.wav'))
        dummy_features = extract_cached(extractor, dummy, out / 'cache', args.deadline)
        atomic_json(out / 'dummy_features.json', dummy_features)
        ordered = list(paths.values())
        random.Random(20260913).shuffle(ordered)
        first = extract_cached(extractor, ordered[:1], out / 'cache', args.deadline)
        if not first:
            raise TimeoutError('No validation feature completed within the budget')
        atomic_json(out / 'reference_parity.json', reference_parity(extractor, ordered[0], first[0]))
        stress_path = args.baseline / 'stress_60s.wav'
        signal = extractor.base.load_audio(ordered[0])
        sf.write(stress_path, np.tile(signal, 960000 // len(signal) + 1)[:960000], 16000)
        try:
            stress = extract_cached(extractor, [stress_path], out / 'stress_cache', args.deadline)
            atomic_json(out / 'stress60.json', {'completed': bool(stress),
                'purpose': '60-second runtime and memory only; repeated source is not validation evidence'})
        except torch.cuda.OutOfMemoryError as error:
            atomic_json(out / 'stress60.json', {'completed': False, 'error': str(error),
                'submission_supported': False})
            torch.cuda.empty_cache()
        features = extract_cached(extractor, ordered, out / 'cache', args.deadline)
    finally:
        extractor.close()
    atomic_json(out / 'features.json', features)
    available = {f['ID'] for f in features}
    completed_labels = [r for r in labels if r['ID'] in available]
    fusion = fit_fusion(features, labels)
    atomic_json(out / 'fusion.json', fusion)
    predictions = {name: [] for name in ('G0', 'G1', 'G2', 'G3', 'G4')}
    for feature in features:
        for name, row in graph_predictions(feature, fusion).items():
            predictions[name].append(row)
    report = {'complete': len(features) == len(labels), 'completed': len(features),
        'planned': len(labels), 'elapsed_seconds': time.time() - start,
        'scope': 'external_source_group_holdout_not_DACON_leaderboard', 'graphs': {}}
    heldout = [r for r in completed_labels if r['split'] == 'holdout']
    heldout_ids = {r['ID'] for r in heldout}
    heldout_groups = {g for r in heldout for g in r['group_id'].split('|')}
    report['holdout'] = {'clips': len(heldout), 'atomic_source_groups': len(heldout_groups),
        'limitation': 'Transforms and mixtures are correlated; source groups are the independent units'}
    weights = {'file_eer': .45, 'voice_eer': .18, 'music_eer': .27,
        'voice_presence_auc': .05, 'music_presence_auc': .05}
    for name, rows in predictions.items():
        validate_predictions(rows, [f['ID'] for f in features])
        write_csv(out / f'{name}_predictions.csv', rows)
        by_id = {r['ID']: r for r in rows if r['ID'] in heldout_ids}
        selected = [by_id[r['ID']] for r in heldout]
        result = official_metrics(heldout, selected) if heldout else {
            'score': None, 'metrics': {k: None for k in weights},
            'invalid': {k: 'no completed holdout examples' for k in weights},
            'partial': {}, 'counts': {k: 0 for k in weights},
            'unknown_counts': {k: 0 for k in weights}, 'coverage': {k: 0.0 for k in weights}}
        # ponytail: partial-label proxy only selects this pilot; it is never an official score.
        result['selection_proxy'] = sum(weight * ((1 - result['metrics'][metric]) if metric.endswith('eer')
            else result['metrics'][metric]) for metric, weight in weights.items()
            if result['metrics'][metric] is not None)
        report['graphs'][name] = result
        print(name, json.dumps(result, ensure_ascii=False), flush=True)
    ranked = sorted(report['graphs'], key=lambda g: report['graphs'][g]['selection_proxy'], reverse=True)
    best = ranked[0]
    improvement = report['graphs'][best]['selection_proxy'] - report['graphs']['G0']['selection_proxy']
    stress_ok = json.loads((out / 'stress60.json').read_text())['completed']
    eligible = report['complete'] and stress_ok and len(heldout_groups) >= 20 and improvement >= .01
    report['selection'] = {'graph': best if eligible else 'G0', 'best_observed_graph': best,
        'proxy_improvement_over_G0': improvement, 'new_submission_supported': eligible,
        'reason': 'complete_pilot_and_proxy_gain_at_least_0.01' if eligible else 'retain_reference_without_sufficient_evidence'}
    records = [json.loads(p.read_text()) for p in (out / 'cache').glob('*.json')]
    timings = [r['seconds'] for r in records]
    report['runtime'] = {'unique_cached_files': len(records), 'total_extraction_seconds': sum(timings),
        'mean_seconds_per_file': float(np.mean(timings)),
        'max_allocated_gib': max(r['max_allocated_gib'] for r in records),
        'max_reserved_gib': max(r['max_reserved_gib'] for r in records),
        'L4_60_minute_compliance': 'unverified; A100 measurements are not L4 evidence'}
    report['runtime']['stress60'] = [
        {k: v for k, v in json.loads(p.read_text()).items() if k != 'feature'}
        for p in (out / 'stress_cache').glob('*.json')]
    atomic_json(out / 'comparison.json', report)
    for name in predictions:
        dummy_rows = [graph_predictions(f, fusion)[name] for f in dummy_features]
        write_csv(out / f'{name}_dummy_submission.csv', dummy_rows)
    atomic_json(out / 'status.json', {'status': 'comparison_finished', 'unix': time.time(),
        'selected_graph': report['selection']['graph'], 'completed': len(features)})
    print('COMPARISON_FINISHED', json.dumps(report['selection']), flush=True)


if __name__ == '__main__':
    main()


Writing /content/dacon236749_v2/run_experiment.py


In [6]:
%%writefile /content/dacon236749_v2/package_submission.py
"""Build a selected graph's offline archive without recompressing model tensors.

The evaluator already supplies the dependencies used here, so requirements.txt
is empty. Colab setup belongs to the experiment notebook, not the submission.
"""

import csv
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath
from typing import TYPE_CHECKING, Final, Literal, TypedDict

from graph_scoring import PredictionRow, SUBMISSION_COLUMNS, validate_predictions

if TYPE_CHECKING:
    from graph_fusion import Fusion


class Selection(TypedDict):
    graph: Literal["G0", "G1", "G2", "G3", "G4"]
    fusion: "Fusion | None"


RUNTIME_FILES: Final = ("graph_features.py", "graph_fusion.py", "graph_scoring.py")
MAX_ZIP_BYTES: Final = 10_000_000_000
MAX_EXPANDED_BYTES: Final = 32_000_000_000
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the selected frozen graph independently on each evaluation audio file."""
import json
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
RUNTIME = ROOT / "model" / "runtime"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(RUNTIME))

from graph_features import Extractor
from graph_fusion import graph_predictions
from graph_scoring import SUBMISSION_COLUMNS, validate_predictions


def main():
    graph = json.loads((RUNTIME / "choice.json").read_text())["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        raise ValueError("Unsupported graph selection")
    fusion = json.loads((RUNTIME / "fusion.json").read_text())
    extractor = Extractor(RUNTIME / "official_baseline.py", include_raw=graph in {"G3", "G4"})
    try:
        base = extractor.base
        columns, sample = base.read_sample_submission(ROOT / "data" / "sample_submission.csv")
        if columns != list(SUBMISSION_COLUMNS):
            raise ValueError("Sample submission columns differ from the official six-column schema")
        paths = base.order_audio_files(base.find_audio_files(ROOT / "data" / "test"), sample)
        rows = []
        for index, path in enumerate(paths, 1):
            prediction = graph_predictions(extractor.extract(path), fusion)[graph]
            rows.extend(validate_predictions([prediction], [path.stem]))
            print(f"PREDICT {index}/{len(paths)} {path.stem}", flush=True)
        rows = validate_predictions(rows, [row["ID"] for row in sample])
        destination = ROOT / "output" / "submission.csv"
        temporary = destination.with_suffix(".csv.tmp")
        base.save_submission(temporary, columns, rows)
        temporary.replace(destination)
    finally:
        extractor.close()


if __name__ == "__main__":
    main()
'''


def _model_members(archive: zipfile.ZipFile) -> list[zipfile.ZipInfo]:
    members = [member for member in archive.infolist() if member.filename.startswith("model/")]
    names = [member.filename for member in members]
    if not members or len(names) != len(set(names)):
        message = "Baseline model assets must be present with unique archive names"
        raise ValueError(message)
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
            message = f"Unsafe baseline member: {member.filename}"
            raise ValueError(message)
        if path.parts[:2] == ("model", "runtime"):
            message = "Baseline already contains reserved model/runtime paths"
            raise ValueError(message)
    return members


def build_submission(original_zip: Path, output_zip: Path, selection: Selection) -> Path:
    """Keep baseline model assets and package one selected graph atomically."""
    if original_zip.resolve() == output_zip.resolve():
        message = "The official baseline archive must not be overwritten"
        raise ValueError(message)
    graph = selection["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        message = f"Unsupported graph selection: {graph}"
        raise ValueError(message)
    fusion = selection.get("fusion")
    if graph == "G4" and fusion is None:
        message = "G4 packaging requires its frozen fusion parameters"
        raise ValueError(message)
    # G0-G3 cannot accidentally execute unrelated learned-head parameters.
    fusion_json = json.dumps(fusion if graph == "G4" else None, allow_nan=False)
    source_directory = Path(__file__).resolve().parent
    runtime = {f"model/runtime/{name}": (source_directory / name).read_bytes() for name in RUNTIME_FILES}
    runtime["model/runtime/fusion.json"] = fusion_json.encode()
    runtime["model/runtime/choice.json"] = json.dumps({"graph": graph}).encode()
    compile(DEPLOY_SCRIPT, "script.py", "exec")
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        with zipfile.ZipFile(original_zip) as source:
            members = _model_members(source)
            baseline = source.read("script.py").decode("utf-8")
            original_base = "BASE_DIR = Path(__file__).resolve().parent"
            if baseline.count(original_base) != 1:
                message = "Official baseline BASE_DIR no longer matches the expected source"
                raise ValueError(message)
            baseline = baseline.replace(original_base, "BASE_DIR = Path(__file__).resolve().parents[2]", 1)
            compile(baseline, "model/runtime/official_baseline.py", "exec")
            runtime["model/runtime/official_baseline.py"] = baseline.encode()
            expanded_bytes = sum(member.file_size for member in members)
            expanded_bytes += sum(map(len, runtime.values())) + len(DEPLOY_SCRIPT.encode())
            if expanded_bytes > MAX_EXPANDED_BYTES or expanded_bytes + 1_000_000 > MAX_ZIP_BYTES:
                message = f"Stored archive would exceed submission limits: {expanded_bytes} bytes"
                raise ValueError(message)
            # Tensor compression costs CPU time without improving the inference result.
            with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as target:
                for member in members:
                    if member.is_dir():
                        target.writestr(member.filename, b"")
                        continue
                    with source.open(member) as incoming, target.open(member.filename, "w", force_zip64=True) as outgoing:
                        shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
                for name, content in runtime.items():
                    target.writestr(name, content)
                target.writestr("script.py", DEPLOY_SCRIPT)
                target.writestr("requirements.txt", b"")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            message = "Built archive exceeds the compressed submission size limit"
            raise ValueError(message)
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
    return output_zip


def verify_packaged(unpacked_dir: Path, deadline: float) -> list[PredictionRow]:
    """Run the real freshly unpacked script and parse its validated predictions.

    The caller provides data/test and data/sample_submission.csv, and compares
    these rows with the notebook's reference predictions. Deadline is Unix time.
    """
    remaining = deadline - time.time()
    if remaining <= 0:
        message = "No experiment time remains for packaged inference"
        raise TimeoutError(message)
    output = unpacked_dir / "output"
    output.mkdir(parents=True, exist_ok=True)
    (output / "submission.csv").unlink(missing_ok=True)
    with (output / "packaged_inference.log").open("w", encoding="utf-8") as log:
        subprocess.run(
            [sys.executable, str((unpacked_dir / "script.py").resolve())],
            cwd=unpacked_dir, stdout=log, stderr=subprocess.STDOUT,
            check=True, timeout=min(remaining, 3600),
        )
    with (unpacked_dir / "data" / "sample_submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        expected_ids = [row["ID"] for row in csv.DictReader(stream)]
    with (output / "submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != list(SUBMISSION_COLUMNS):
            message = "Packaged output must have the exact official six-column header"
            raise ValueError(message)
        rows = list(reader)
    return validate_predictions(rows, expected_ids)


Writing /content/dacon236749_v2/package_submission.py


In [7]:
%%writefile /content/dacon236749_v2/ast_multitask.py
"""Pinned AST encoder with five independently trainable DACON heads."""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Final, Literal, assert_never

import numpy as np
import torch
import torch.nn.functional as functional
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import ASTConfig, ASTFeatureExtractor, ASTModel

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = 160_000
PROCESSOR_CONTRACT: Final = {
    "sampling_rate": SAMPLE_RATE,
    "num_mel_bins": 128,
    "max_length": 1024,
    "mean": -4.2677393,
    "std": 4.5689974,
}
TrainMode = Literal["frozen", "last_block"]


class ASTContractError(ValueError):
    """An offline AST asset or input violates the pinned model contract."""


def _validate_processor(processor: ASTFeatureExtractor) -> None:
    for name, expected in PROCESSOR_CONTRACT.items():
        actual = getattr(processor, name, None)
        if not isinstance(actual, int | float) or not math.isclose(float(actual), expected):
            raise ASTContractError("AST processor contract mismatch")


def load_processor(local_dir: Path) -> ASTFeatureExtractor:
    """Load the pinned processor locally and fail closed on configuration drift."""
    processor = ASTFeatureExtractor.from_pretrained(local_dir, local_files_only=True)
    _validate_processor(processor)
    return processor


class ASTMultiTask(nn.Module):
    """AST pooler followed by five independent binary heads."""

    def __init__(self, backbone: ASTModel, mode: TrainMode) -> None:
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleList(nn.Linear(backbone.config.hidden_size, 1) for _ in PREDICTION_COLUMNS)
        self.mode = mode
        self.configure_trainable(mode)

    @classmethod
    def from_pretrained(cls, local_dir: Path, mode: TrainMode) -> ASTMultiTask:
        """Load only supplied local AST weights; AudioSet classifier logits are unused."""
        load_processor(local_dir)
        backbone = ASTModel.from_pretrained(
            local_dir,
            local_files_only=True,
            attn_implementation="sdpa",
        )
        return cls(backbone, mode)

    def configure_trainable(self, mode: TrainMode) -> None:
        """Expose exactly the parameters owned by the selected training mode."""
        for parameter in self.parameters():
            parameter.requires_grad_(False)
        for parameter in self.heads.parameters():
            parameter.requires_grad_(True)
        match mode:
            case "frozen":
                pass
            case "last_block":
                for parameter in self.backbone.encoder.layer[-1].parameters():
                    parameter.requires_grad_(True)
                for parameter in self.backbone.layernorm.parameters():
                    parameter.requires_grad_(True)
            case unreachable:
                assert_never(unreachable)
        self.mode = mode

    def train(self, mode: bool = True) -> ASTMultiTask:
        """Keep frozen encoder blocks deterministic while training selected layers."""
        super().train(mode)
        if mode:
            self.backbone.eval()
            if self.mode == "last_block":
                self.backbone.encoder.layer[-1].train()
        return self

    def encode(self, input_values: torch.Tensor) -> torch.Tensor:
        """Return the final-normalized CLS/distillation-token pooler output."""
        return self.backbone(input_values=input_values).pooler_output

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        pooled = self.encode(input_values)
        return torch.cat([head(pooled) for head in self.heads], dim=-1)


def _eligible_mask(targets: torch.Tensor) -> torch.Tensor:
    if targets.ndim != 2 or targets.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST targets must have five columns")
    known = torch.isfinite(targets)
    if torch.any(known & (targets != 0) & (targets != 1)):
        raise ASTContractError("AST targets must be binary or NaN")
    eligible = known.clone()
    eligible[:, 1] &= known[:, 3] & (targets[:, 3] == 1)
    eligible[:, 2] &= known[:, 4] & (targets[:, 4] == 1)
    return eligible


def compute_pos_weights(targets: torch.Tensor) -> torch.Tensor:
    """Compute bounded training-only positive weights over eligible truth."""
    eligible = _eligible_mask(targets)
    weights = torch.ones(len(PREDICTION_COLUMNS), dtype=targets.dtype, device=targets.device)
    for index in range(len(PREDICTION_COLUMNS)):
        selected = targets[:, index][eligible[:, index]]
        positives = torch.count_nonzero(selected == 1)
        negatives = torch.count_nonzero(selected == 0)
        if positives > 0 and negatives > 0:
            weights[index] = (negatives / positives).clamp(0.25, 4.0)
    return weights


def masked_multitask_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weights: torch.Tensor,
) -> torch.Tensor:
    """Mean five independently normalized BCE heads over known eligible labels."""
    if logits.shape != targets.shape or pos_weights.shape != (len(PREDICTION_COLUMNS),):
        raise ASTContractError("AST loss input contract mismatch")
    eligible = _eligible_mask(targets)
    losses: list[torch.Tensor] = []
    safe_targets = torch.nan_to_num(targets)
    for index in range(len(PREDICTION_COLUMNS)):
        mask = eligible[:, index]
        if torch.any(mask):
            losses.append(functional.binary_cross_entropy_with_logits(
                logits[:, index][mask],
                safe_targets[:, index][mask],
                pos_weight=pos_weights[index],
            ))
    if not losses:
        raise ASTContractError("AST batch has no eligible labels")
    return torch.stack(losses).mean()


def split_audio(
    audio: np.ndarray,
    sample_rate: int = SAMPLE_RATE,
    window_samples: int = WINDOW_SAMPLES,
) -> tuple[np.ndarray, ...]:
    """Cover one mono file with deterministic, non-overlapping windows including its tail."""
    if sample_rate != SAMPLE_RATE or window_samples <= 0 or audio.ndim != 1 or audio.size == 0:
        raise ASTContractError("AST audio input contract mismatch")
    if not np.isfinite(audio).all():
        raise ASTContractError("AST audio contains non-finite samples")
    return tuple(audio[start:start + window_samples] for start in range(0, audio.size, window_samples))


def prepare_inputs(
    processor: ASTFeatureExtractor,
    windows: tuple[np.ndarray, ...],
) -> torch.Tensor:
    """Apply the pinned AST feature extractor to one file's windows."""
    _validate_processor(processor)
    if not windows:
        raise ASTContractError("AST requires at least one audio window")
    batch = processor(list(windows), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    return batch["input_values"]


def aggregate_window_logits(logits: torch.Tensor) -> torch.Tensor:
    """Mean all window probabilities from exactly one file."""
    if logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST window logits contract mismatch")
    return torch.sigmoid(logits.float()).mean(dim=0)


def predict_file(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    audio: np.ndarray,
) -> torch.Tensor:
    """Predict one file without reading or retaining another file's state."""
    inputs = prepare_inputs(processor, split_audio(audio)).to(next(model.parameters()).device)
    with torch.inference_mode():
        return aggregate_window_logits(model(inputs)).cpu()


def export_checkpoint(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    directory: Path,
) -> Path:
    """Export a self-contained inference checkpoint without optimizer state."""
    _validate_processor(processor)
    directory.mkdir(parents=True, exist_ok=True)
    model.backbone.config.to_json_file(directory / "config.json")
    processor.save_pretrained(directory)
    metadata = {
        "format_version": 1,
        "mode": model.mode,
        "columns": list(PREDICTION_COLUMNS),
        "hidden_size": model.backbone.config.hidden_size,
        "processor_contract": PROCESSOR_CONTRACT,
    }
    metadata_path = directory / "multitask_config.json"
    temporary_metadata = metadata_path.with_suffix(".json.tmp")
    temporary_metadata.write_text(json.dumps(metadata, sort_keys=True), encoding="utf-8")
    temporary_metadata.replace(metadata_path)
    weights_path = directory / "model.safetensors"
    temporary_weights = weights_path.with_suffix(".safetensors.tmp")
    save_file(
        {name: value.detach().cpu().contiguous() for name, value in model.state_dict().items()},
        temporary_weights,
    )
    temporary_weights.replace(weights_path)
    return directory


def load_checkpoint(
    directory: Path,
    mode: TrainMode,
) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load a self-contained checkpoint and reject processor/head contract drift."""
    metadata_path = directory / "multitask_config.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "format_version": 1,
        "mode": mode,
        "columns": list(PREDICTION_COLUMNS),
    }
    if any(metadata.get(name) != value for name, value in expected.items()):
        raise ASTContractError("AST checkpoint contract mismatch")
    processor = load_processor(directory)
    config = ASTConfig.from_json_file(directory / "config.json")
    if metadata.get("hidden_size") != config.hidden_size:
        raise ASTContractError("AST checkpoint contract mismatch")
    model = ASTMultiTask(ASTModel(config), mode)
    model.load_state_dict(load_file(directory / "model.safetensors"), strict=True)
    return model, processor


Writing /content/dacon236749_v2/ast_multitask.py


In [8]:
%%writefile /content/dacon236749_v2/v2_scoring.py
"""Exact ROC diagnostics and development-only model selection for V2."""

from __future__ import annotations

import argparse, csv, json, random
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Final

from graph_scoring import official_metrics, validate_predictions

Scalar = str | int | float | None
Row = Mapping[str, Scalar]
PredictionRows = Mapping[str, Sequence[Row]]
WEIGHTS: Final = {"file_eer": .45, "voice_eer": .18, "music_eer": .27, "voice_presence_auc": .05, "music_presence_auc": .05}
EER_SPECS: Final = (("file", "FILE_FAKE", "FILE_FAKE_PROB", None), ("voice", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"), ("music", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"))


def _label(row: Row, name: str) -> int | None:
    value = row.get(name)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise ValueError(f"invalid binary label: {name}")
    return int(value)


def _role(row: Row) -> str:
    role = row.get("role", row.get("split"))
    if not isinstance(role, str) or not role:
        raise ValueError("manifest rows require role or split")
    return role


def _prediction_map(rows: Sequence[Row], manifest: Sequence[Row]) -> dict[str, Row]:
    expected = [str(row["ID"]) for row in manifest]
    parsed = validate_predictions(rows, expected)
    return {row["ID"]: row for row in parsed}


def _eligible(rows: Sequence[Row], predictions: Mapping[str, Row], target: str,
              score: str, mask: str | None) -> tuple[list[int], list[float], int]:
    targets: list[int] = []
    scores: list[float] = []
    unknown = 0
    for row in rows:
        if mask is not None:
            present = _label(row, mask)
            if present is None:
                unknown += 1
                continue
            if present == 0:
                continue
        truth = _label(row, target)
        if truth is None:
            unknown += 1
            continue
        targets.append(truth)
        scores.append(float(predictions[str(row["ID"])][score]))
    return targets, scores, unknown


def _roc(targets: Sequence[int], scores: Sequence[float]) -> dict[str, Scalar] | None:
    positives = sum(targets)
    negatives = len(targets) - positives
    if not positives or not negatives:
        return None
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    points: list[tuple[float | None, int, int]] = [(None, 0, 0)]
    tp = fp = 0
    index = 0
    while index < len(ranked):
        threshold = ranked[index][0]
        while index < len(ranked) and ranked[index][0] == threshold:
            tp += ranked[index][1]
            fp += 1 - ranked[index][1]
            index += 1
        points.append((threshold, tp, fp))
    def distance(point: tuple[float | None, int, int]) -> float:
        return abs(point[2] / negatives - (1 - point[1] / positives))
    threshold, tp, fp = min(points, key=distance)
    fpr, fnr = fp / negatives, 1 - tp / positives
    auc = sum((right[2] / negatives - left[2] / negatives) *
              (right[1] / positives + left[1] / positives) / 2
              for left, right in zip(points, points[1:]))
    return {"eer": (fpr + fnr) / 2, "auc": auc, "threshold": threshold,
            "fpr": fpr, "fnr": fnr, "fp": fp, "fn": positives - tp,
            "tp": tp, "tn": negatives - fp, "n": len(targets)}


def _detail_report(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, dict[str, Scalar]]:
    details: dict[str, dict[str, Scalar]] = {}
    for name, target, score, mask in EER_SPECS:
        targets, scores, unknown = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        details[name] = {"unknown_relevant": unknown, "n": len(targets),
                         "point": point}
    return details


def _pair_changes(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row],
                  target: str, score: str, mask: str | None) -> dict[str, int]:
    labels, scores_a, _ = _eligible(rows, baseline, target, score, mask)
    _, scores_b, _ = _eligible(rows, candidate, target, score, mask)
    positives = [index for index, value in enumerate(labels) if value == 1]
    negatives = [index for index, value in enumerate(labels) if value == 0]
    flips = worsened = 0
    for positive in positives:
        for negative in negatives:
            before = scores_a[positive] - scores_a[negative]
            after = scores_b[positive] - scores_b[negative]
            if (before > 0 > after) or (before < 0 < after):
                flips += 1
                worsened += int(before > 0 > after)
    return {"pairs": len(positives) * len(negatives), "flips": flips,
            "positive_to_negative_worsened": worsened}


def audit_forensics(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Report all available roles without converting a partial proxy into a score."""
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for forensics")
    roles: dict[str, list[Row]] = defaultdict(list)
    for row in manifest_rows:
        roles[_role(row)].append(row)
    report: dict[str, object] = {"metric": "nearest_abs_fpr_minus_fnr", "roles": {}}
    for role, rows in sorted(roles.items()):
        models: dict[str, object] = {}
        for name, prediction in maps.items():
            models[name] = {"official_proxy_only": official_metrics(rows, [prediction[str(row["ID"])] for row in rows]),
                            "roc": _detail_report(rows, prediction)}
        comparisons: dict[str, object] = {}
        for name, prediction in maps.items():
            if name == "G0":
                continue
            per_head: dict[str, object] = {}
            for head, target, score, mask in EER_SPECS:
                base = models["G0"]["roc"][head]["point"]
                changed = models[name]["roc"][head]["point"]
                per_head[head] = {"baseline": base, "candidate": changed,
                                  "auc_delta": None if base is None or changed is None else changed["auc"] - base["auc"],
                                  "pair_changes": _pair_changes(rows, maps["G0"], prediction, target, score, mask)}
            comparisons[name] = per_head
        report["roles"][role] = {"models": models, "comparisons": comparisons}
    return report


def _proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, object]:
    metrics = official_metrics(rows, [predictions[str(row["ID"])] for row in rows])
    usable = {name: weight for name, weight in WEIGHTS.items() if metrics["metrics"][name] is not None}
    total = sum(usable.values())
    values = metrics["metrics"]
    proxy = None if not total else sum(weight / total * ((1 - float(values[name])) if name.endswith("eer") else float(values[name]))
                                      for name, weight in usable.items())
    return {"value": proxy, "proxy_weights": usable, "metric_report": metrics,
            "is_partial": bool(metrics["partial"])}


def _resampled_proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> float | None:
    """Score a bootstrap sample whose repeated IDs cannot enter CSV validation."""
    values: dict[str, float] = {}
    for metric, target, score, mask in EER_SPECS:
        targets, scores, _ = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_eer"] = point["eer"]
    for metric, target, score in (("voice_presence", "VOICE_PRESENT", "VOICE_PRESENT_PROB"),
                                  ("music_presence", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB")):
        targets, scores, _ = _eligible(rows, predictions, target, score, None)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_auc"] = point["auc"]
    usable = {name: weight for name, weight in WEIGHTS.items() if name in values}
    total = sum(usable.values())
    return None if not total else sum(weight / total * ((1 - values[name]) if name.endswith("eer") else values[name])
                                     for name, weight in usable.items())


def _components(rows: Sequence[Row]) -> list[list[int]]:
    parent: dict[str, str] = {}
    def find(value: str) -> str:
        parent.setdefault(value, value)
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value
    def join(left: str, right: str) -> None:
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left
    for row in rows:
        raw_group = row.get("group_id")
        if not isinstance(raw_group, str) or not raw_group:
            raise ValueError("manifest rows require nonempty group_id constituents")
        groups = raw_group.split("|")
        if any(not group for group in groups):
            raise ValueError("manifest rows require nonempty group_id constituents")
        for group in groups[1:]:
            join(groups[0], group)
    units: dict[str, list[int]] = defaultdict(list)
    for index, row in enumerate(rows):
        units[find(str(row["group_id"]).split("|")[0])].append(index)
    return list(units.values())


def _bootstrap(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> dict[str, object]:
    units = _components(rows)
    rng = random.Random(20260913)
    deltas: list[float] = []
    for _ in range(2000):
        sampled = [row for unit in (rng.choice(units) for _ in units) for index in unit for row in [rows[index]]]
        base, changed = _resampled_proxy(sampled, baseline), _resampled_proxy(sampled, candidate)
        if base is not None and changed is not None:
            deltas.append(changed - base)
    deltas.sort()
    return {"seed": 20260913, "resamples": 2000, "lineage_components": len(units),
            "valid_resamples": len(deltas), "uncomputable_resamples": 2000 - len(deltas),
            "win_rate": None if not deltas else sum(delta > 0 for delta in deltas) / len(deltas),
            "p10_delta": None if not deltas else deltas[max(0, (len(deltas) - 1) // 10)]}


def select_candidate(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Select a V2 candidate only from the dedicated development selector role."""
    if any(_role(row) != "dev_select" for row in manifest_rows):
        raise ValueError("selection accepts development roles only")
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for selection")
    models: dict[str, dict[str, object]] = {"G0": _proxy(manifest_rows, maps["G0"])}
    candidates = [name for name in ("AST_FROZEN", "AST_LAST_BLOCK") if name in maps]
    for name in candidates:
        current = _proxy(manifest_rows, maps[name])
        bootstrap = _bootstrap(manifest_rows, maps["G0"], maps[name])
        current["bootstrap"] = bootstrap
        base_metrics = models["G0"]["metric_report"]["metrics"]
        new_metrics = current["metric_report"]["metrics"]
        regression = any(float(new_metrics[key]) > float(base_metrics[key]) + .02 for key in WEIGHTS if key.endswith("eer") and new_metrics[key] is not None and base_metrics[key] is not None)
        companion = any(_roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2])["auc"] < _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2])["auc"] - .01 for _, target, score, mask in EER_SPECS if _roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2]) is not None and _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2]) is not None)
        gain = None if current["value"] is None or models["G0"]["value"] is None else float(current["value"]) - float(models["G0"]["value"])
        gates = {"proxy_gain": gain is not None and gain >= .005,
                 "bootstrap_complete": bootstrap["valid_resamples"] == 2000,
                 "bootstrap_win_rate": bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
                 "bootstrap_p10": bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005,
                 "eer_regression": not regression, "companion_auc_regression": not companion}
        current["gates"] = {**gates, "proxy_gain_delta": gain, "passed": all(gates.values())}
        models[name] = current
    passed = [name for name in candidates if models[name]["gates"]["passed"]]
    selected = max(passed, key=lambda name: (models[name]["value"], -candidates.index(name)), default="G0")
    failed = {name: [gate for gate, value in models[name]["gates"].items() if value is False]
              for name in candidates if name not in passed}
    return {"selected": selected, "reason": "all_gates_passed" if selected != "G0" else "no_candidate_passed_all_gates",
            "failed_gates": failed, "models": models}


def _read_csv(path: Path) -> list[dict[str, Scalar]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return [dict(row) for row in csv.DictReader(stream)]


def _load_predictions(directory: Path, manifest: Sequence[Row]) -> dict[str, list[Row]]:
    files = {name: directory / f"{name}_predictions.csv" for name in ("G0", "G1", "G2", "G3", "G4")}
    if all(path.is_file() for path in files.values()):
        return {name: _read_csv(path) for name, path in files.items()}
    from graph_fusion import graph_predictions
    features = json.loads((directory / "features.json").read_text())
    fusion = json.loads((directory / "fusion.json").read_text())
    by_id = {feature["ID"]: graph_predictions(feature, fusion) for feature in features}
    return {name: [by_id[str(row["ID"])][name] for row in manifest] for name in files}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("diagnose")
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--predictions", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    manifest = _read_csv(args.manifest)
    report = audit_forensics(manifest, _load_predictions(args.predictions, manifest))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/v2_scoring.py


In [9]:
%%writefile /content/dacon236749_v2/train_v2.py
"""Bounded AST training on development data only."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch

from graph_features import atomic_json, digest
from graph_scoring import LABEL_COLUMNS, PREDICTION_COLUMNS, official_metrics
from v2_inference import load_audio as _load_audio

SEED = 20260913
RESERVE_SECONDS = 20 * 60
CONFIGS = {
    "AST_FROZEN": {"mode": "frozen", "epochs": 8, "batch_size": 16,
                   "accumulation": 1, "head_lr": 3e-4, "backbone_lr": 0.0},
    "AST_LAST_BLOCK": {"mode": "last_block", "epochs": 6, "batch_size": 8,
                       "accumulation": 2, "head_lr": 1e-4, "backbone_lr": 1e-5},
}


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False


def _json_hash(value: object) -> str:
    raw = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()


def read_development_manifest(path: Path, data_root: Path) -> list[dict[str, str]]:
    """Reject locked-test inputs before any audio file is opened."""
    resolved = path.resolve()
    if path.name != "manifest.csv" or path.parent.name != "development" or "locked_test" in resolved.parts:
        raise ValueError("training accepts development manifest only")
    with path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or len({row["ID"] for row in rows}) != len(rows):
        raise ValueError("development manifest must contain unique rows")
    for row in rows:
        role = row.get("role", row.get("split", ""))
        if role not in {"dev_train", "dev_select"}:
            raise ValueError("training accepts development roles only")
        audio = (data_root / row["path"]).resolve()
        if not audio.is_relative_to(data_root.resolve()):
            raise ValueError("manifest audio path escapes data root")
    return rows


def _train_window(audio: np.ndarray, identifier: str, epoch: int) -> np.ndarray:
    rng = random.Random(f"{SEED}:{epoch}:{identifier}")
    if len(audio) > 160000:
        start = rng.randrange(len(audio) - 160000 + 1)
        audio = audio[start:start + 160000]
    gain = 10 ** (rng.uniform(-6.0, 6.0) / 20)
    return np.clip(audio * gain, -1.0, 1.0).astype(np.float32)


class FrozenEmbeddingCache:
    """Content-address frozen pooler values; provenance drift creates a new key."""

    def __init__(self, directory: Path, model_sha: str, processor_signature: str):
        self.directory = directory
        self.signature = _json_hash({"model_sha256": model_sha,
                                     "processor": processor_signature,
                                     "augmentation": "seed20260913-fixed-crop-gain-v1"})
        self.misses = 0

    def path_for(self, audio_path: Path, identifier: str) -> Path:
        key = hashlib.sha256(f"{digest(audio_path)}:{identifier}:{self.signature}".encode()).hexdigest()
        return self.directory / f"{key}.pt"

    def get(self, audio_path: Path, identifier: str) -> torch.Tensor | None:
        path = self.path_for(audio_path, identifier)
        if not path.exists():
            self.misses += 1
            if self.misses % 25 == 0:
                print(f"FROZEN_CACHE {self.misses} embeddings", flush=True)
            return None
        record = torch.load(path, map_location="cpu", weights_only=True)
        if record["signature"] != self.signature or record["audio_sha256"] != digest(audio_path):
            raise ValueError("frozen embedding cache provenance mismatch")
        return record["embedding"]

    def put(self, audio_path: Path, identifier: str, embedding: torch.Tensor) -> None:
        path = self.path_for(audio_path, identifier)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp")
        torch.save({"signature": self.signature, "audio_sha256": digest(audio_path),
                    "embedding": embedding.detach().cpu()}, temporary)
        temporary.replace(path)


def _targets(rows: list[dict[str, str]], device: torch.device) -> torch.Tensor:
    values = [[float(row[column]) if row.get(column, "") != "" else math.nan
               for column in LABEL_COLUMNS] for row in rows]
    return torch.tensor(values, dtype=torch.float32, device=device)


def _proxy(report: dict) -> float:
    weights = {"file_eer": .45, "voice_eer": .18, "music_eer": .27,
               "voice_presence_auc": .05, "music_presence_auc": .05}
    available = {name: value for name, value in report["metrics"].items() if value is not None}
    total = sum(weights[name] for name in available)
    if not total:
        return float("-inf")
    return sum(weights[name] * ((1 - value) if name.endswith("eer") else value)
               for name, value in available.items()) / total


def _trainable_state(model: torch.nn.Module) -> dict[str, torch.Tensor]:
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items() if name in names}


def _rng_state() -> dict:
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def _restore_rng(state: dict) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def save_training_checkpoint(path: Path, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                             epoch: int, config_hash: str, reports: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save({"epoch": epoch, "config_hash": config_hash, "model": _trainable_state(model),
                "optimizer": optimizer.state_dict(), "rng": _rng_state(), "reports": reports}, temporary)
    temporary.replace(path)


def load_training_checkpoint(path: Path, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                             config_hash: str) -> tuple[int, list[dict]]:
    state = torch.load(path, map_location="cpu", weights_only=False)
    if state["config_hash"] != config_hash:
        raise ValueError("checkpoint configuration mismatch")
    missing, unexpected = model.load_state_dict(state["model"], strict=False)
    trainable = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    if unexpected or trainable.intersection(missing):
        raise ValueError("checkpoint trainable parameters do not match model")
    optimizer.load_state_dict(state["optimizer"])
    _restore_rng(state["rng"])
    return int(state["epoch"]), list(state["reports"])


def _optimizer(model: torch.nn.Module, config: dict) -> torch.optim.AdamW:
    head_ids = {id(parameter) for parameter in model.heads.parameters()}
    heads = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) in head_ids]
    backbone = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) not in head_ids]
    groups = [{"params": heads, "lr": config["head_lr"]}]
    if backbone:
        groups.append({"params": backbone, "lr": config["backbone_lr"]})
    return torch.optim.AdamW(groups, weight_decay=.01)


def _batch_inputs(rows: list[dict[str, str]], root: Path, processor, epoch: int, device: torch.device):
    from ast_multitask import prepare_inputs
    windows = [_train_window(_load_audio(root / row["path"]), row["ID"], epoch) for row in rows]
    return prepare_inputs(processor, tuple(windows)).to(device), _targets(rows, device)


def _frozen_logits(model, processor, rows: list[dict[str, str]], root: Path,
                   cache: FrozenEmbeddingCache, device: torch.device) -> torch.Tensor:
    from ast_multitask import prepare_inputs
    embeddings = []
    for row in rows:
        path = root / row["path"]
        embedding = cache.get(path, row["ID"])
        if embedding is None:
            window = _train_window(_load_audio(path), row["ID"], 0)
            with torch.no_grad():
                embedding = model.encode(prepare_inputs(processor, (window,)).to(device))[0].cpu()
            cache.put(path, row["ID"], embedding)
        embeddings.append(embedding)
    pooled = torch.stack(embeddings).to(device)
    return torch.cat([head(pooled) for head in model.heads], dim=-1)


@torch.no_grad()
def predict_rows(model, processor, rows: list[dict[str, str]], root: Path,
                 device: torch.device,
                 deadline: float | None = None,
                 cache: FrozenEmbeddingCache | None = None) -> list[dict[str, str | float]]:
    from ast_multitask import aggregate_window_logits, prepare_inputs, split_audio
    model.eval()
    predictions = []
    for row in rows:
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError("prediction deadline reached before completing a file")
        path = root / row["path"]
        embeddings = cache.get(path, row["ID"] + "#eval") if cache else None
        if embeddings is None:
            windows = split_audio(_load_audio(path))
            if cache:
                embeddings = model.encode(prepare_inputs(processor, windows).to(device)).cpu()
                cache.put(path, row["ID"] + "#eval", embeddings)
        logits = (torch.cat([head(embeddings.to(device)) for head in model.heads], dim=-1)
                  if cache else model(prepare_inputs(processor, windows).to(device)))
        probabilities = aggregate_window_logits(logits).float().cpu().tolist()
        predictions.append({"ID": row["ID"], **dict(zip(PREDICTION_COLUMNS, probabilities, strict=True))})
    return predictions


def train_candidate(name: str, model, processor, rows: list[dict[str, str]], data_root: Path,
                    output: Path, deadline: float, manifest_hash: str,
                    asset_model_sha: str) -> tuple[object, dict]:
    from ast_multitask import compute_pos_weights, export_checkpoint, masked_multitask_loss
    config = {**CONFIGS[name], "seed": SEED, "manifest_sha256": manifest_hash,
              "asset_model_sha256": asset_model_sha,
              "weight_decay": .01, "gradient_clip": 1.0}
    config_hash = _json_hash(config)
    if time.time() >= deadline - RESERVE_SECONDS:
        raise TimeoutError(f"not enough time to start {name}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = _optimizer(model, config)
    train = [row for row in rows if row.get("role", row.get("split")) == "dev_train"]
    select = [row for row in rows if row.get("role", row.get("split")) == "dev_select"]
    pos_weights = compute_pos_weights(_targets(train, device))
    processor_signature = _json_hash({name: getattr(processor, name) for name in
                                      ("sampling_rate", "num_mel_bins", "max_length", "mean", "std")})
    frozen_cache = FrozenEmbeddingCache(output / name / "pooler_cache", asset_model_sha,
                                        processor_signature) if name == "AST_FROZEN" else None
    checkpoint = output / name / "last.pt"
    start_epoch, reports = (0, [])
    if checkpoint.exists():
        start_epoch, reports = load_training_checkpoint(checkpoint, model, optimizer, config_hash)
    best_proxy = max((report["proxy"] for report in reports), default=float("-inf"))
    best_state = None
    if start_epoch and (output / name / "best.pt").exists():
        best_state = torch.load(output / name / "best.pt", map_location="cpu", weights_only=True)
    for epoch in range(start_epoch + 1, config["epochs"] + 1):
        if time.time() >= deadline - RESERVE_SECONDS:
            break
        started = time.perf_counter()
        model.train()
        before_epoch = {key: value.clone() for key, value in _trainable_state(model).items()}
        optimizer.zero_grad(set_to_none=True)
        shuffled = list(train)
        random.Random(SEED + epoch).shuffle(shuffled)
        losses = []
        gradient_norms = []
        head_gradient_norms = []
        backbone_gradient_norms = []
        for batch_index in range(0, len(shuffled), config["batch_size"]):
            if time.time() >= deadline - RESERVE_SECONDS:
                raise TimeoutError(f"deadline reached during {name}; incomplete epoch was discarded")
            batch = shuffled[batch_index:batch_index + config["batch_size"]]
            targets = _targets(batch, device)
            amp = torch.autocast("cuda", dtype=torch.bfloat16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
            with amp:
                logits = (_frozen_logits(model, processor, batch, data_root, frozen_cache, device)
                          if frozen_cache else model(_batch_inputs(batch, data_root, processor, 0, device)[0]))
                loss = masked_multitask_loss(logits, targets, pos_weights) / config["accumulation"]
            loss.backward()
            losses.append(float(loss.detach()) * config["accumulation"])
            step = (batch_index // config["batch_size"] + 1) % config["accumulation"] == 0
            last = batch_index + config["batch_size"] >= len(shuffled)
            if step or last:
                heads = [p for p in model.heads.parameters() if p.requires_grad]
                backbone = [p for parameter_name, p in model.named_parameters()
                            if p.requires_grad and not parameter_name.startswith("heads.")]
                head_gradient_norms.append(float(torch.nn.utils.clip_grad_norm_(heads, float("inf"))))
                if backbone:
                    backbone_gradient_norms.append(float(torch.nn.utils.clip_grad_norm_(backbone, float("inf"))))
                norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                gradient_norms.append(float(norm))
                if not math.isfinite(gradient_norms[-1]):
                    raise FloatingPointError(f"non-finite gradient in {name}")
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        if max(head_gradient_norms, default=0.0) <= 0 or (
            name == "AST_LAST_BLOCK" and max(backbone_gradient_norms, default=0.0) <= 0
        ):
            raise FloatingPointError(f"intended trainable path received no gradient in {name}")
        predictions = predict_rows(model, processor, select, data_root, device,
                                   deadline=deadline - RESERVE_SECONDS, cache=frozen_cache)
        metric_report = official_metrics(select, predictions)
        proxy = _proxy(metric_report)
        report = {"epoch": epoch, "mean_loss": float(np.mean(losses)), "proxy": proxy,
                  "metric_report": metric_report, "seconds": time.perf_counter() - started,
                  "max_gradient_norm": max(gradient_norms, default=0.0),
                  "max_head_gradient_norm": max(head_gradient_norms, default=0.0),
                  "max_backbone_gradient_norm": max(backbone_gradient_norms, default=0.0),
                  "trainable_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items()),
                  "head_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items() if key.startswith("heads.")),
                  "backbone_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items() if not key.startswith("heads."))}
        reports.append(report)
        if proxy > best_proxy:
            best_proxy, best_state = proxy, _trainable_state(model)
            best_path = output / name / "best.pt"
            best_path.parent.mkdir(parents=True, exist_ok=True)
            temporary = best_path.with_suffix(".tmp")
            torch.save(best_state, temporary)
            temporary.replace(best_path)
        save_training_checkpoint(checkpoint, model, optimizer, epoch, config_hash, reports)
        atomic_json(output / name / f"epoch_{epoch:03d}.json", report)
        print(f"TRAIN_EPOCH {name} {epoch}/{config['epochs']} "
              f"loss={report['mean_loss']:.6f} proxy={proxy:.6f} "
              f"head_grad={report['max_head_gradient_norm']:.6f} "
              f"backbone_grad={report['max_backbone_gradient_norm']:.6f}", flush=True)
    if best_state is None:
        raise TimeoutError(f"{name} completed no epoch")
    model.load_state_dict(best_state, strict=False)
    exported = output / name / "best"
    export_checkpoint(model, processor, exported)
    predictions = predict_rows(model, processor, select, data_root, device,
                               deadline=deadline - RESERVE_SECONDS, cache=frozen_cache)
    changed = max(report.get("trainable_absolute_update", 0.0) for report in reports)
    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    result = {"candidate": name, "checkpoint": str(exported), "config": config,
              "config_hash": config_hash, "best_proxy": best_proxy, "epochs_completed": len(reports),
              "predictions": predictions, "reports": reports, "trainable_names": trainable_names,
              "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
              "trainable_absolute_update": changed,
              "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else 0.0,
              "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3 if device.type == "cuda" else 0.0}
    prediction_path = output / name / "dev_predictions.csv"
    with prediction_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["ID", *PREDICTION_COLUMNS])
        writer.writeheader()
        writer.writerows(predictions)
    atomic_json(output / name / "report.json", result)
    return model, result


def train_candidates(ast_assets: Path, manifest_path: Path, data_root: Path,
                     output: Path, deadline: float) -> dict:
    from ast_multitask import ASTMultiTask, load_processor
    seed_everything()
    rows = read_development_manifest(manifest_path, data_root)
    processor = load_processor(ast_assets)
    asset_model_sha = digest(ast_assets / "model.safetensors")
    results = {}
    frozen, results["AST_FROZEN"] = train_candidate(
        "AST_FROZEN", ASTMultiTask.from_pretrained(ast_assets, "frozen"), processor,
        rows, data_root, output, deadline, digest(manifest_path), asset_model_sha)
    if time.time() < deadline - RESERVE_SECONDS:
        last = ASTMultiTask.from_pretrained(ast_assets, "last_block")
        last.heads.load_state_dict(frozen.heads.state_dict())
        _, results["AST_LAST_BLOCK"] = train_candidate(
            "AST_LAST_BLOCK", last, processor, rows, data_root, output, deadline,
            digest(manifest_path), asset_model_sha)
    else:
        results["AST_LAST_BLOCK"] = {"candidate": "AST_LAST_BLOCK", "status": "skipped_deadline"}
    report = {"candidates": results, "manifest_sha256": digest(manifest_path),
              "finished_unix": time.time()}
    atomic_json(output / "training_report.json", report)
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--assets", type=Path, required=True)
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    args = parser.parse_args()
    train_candidates(args.assets, args.manifest, args.data_root, args.output, args.deadline)


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/train_v2.py


In [10]:
%%writefile /content/dacon236749_v2/v2_inference.py
"""Offline, file-local inference for the self-contained v2 AST submission."""

from __future__ import annotations

import csv
import json
import math
import os
from pathlib import Path
from typing import Final

import numpy as np
import soundfile as soundfile
import torch
from scipy.signal import resample_poly
from transformers import ASTFeatureExtractor

from ast_multitask import PREDICTION_COLUMNS, SAMPLE_RATE, ASTMultiTask, load_checkpoint, predict_file

SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
SUPPORTED_AUDIO_SUFFIXES: Final = frozenset({".wav", ".flac", ".mp3"})


class InferenceContractError(ValueError):
    """The evaluator input or local inference asset violates the submission contract."""


def load_audio(path: Path) -> np.ndarray:
    """Decode one evaluator file to finite mono 16 kHz float32 PCM without a network codec."""
    try:
        waveform, sample_rate = soundfile.read(path, dtype="float32", always_2d=True)
    except (OSError, RuntimeError) as error:
        raise InferenceContractError(f"Cannot decode audio file: {path.name}") from error
    if waveform.size == 0 or sample_rate <= 0 or not np.isfinite(waveform).all():
        raise InferenceContractError(f"Invalid audio file: {path.name}")
    mono = waveform.mean(axis=1, dtype=np.float32)
    if sample_rate != SAMPLE_RATE:
        divisor = math.gcd(sample_rate, SAMPLE_RATE)
        mono = resample_poly(mono, SAMPLE_RATE // divisor, sample_rate // divisor).astype(np.float32)
    if mono.size == 0 or not np.isfinite(mono).all():
        raise InferenceContractError(f"Invalid resampled audio file: {path.name}")
    return mono


def _sample_ids(path: Path) -> list[str]:
    """Parse the evaluator-owned ID order once at the CSV trust boundary."""
    try:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != list(SUBMISSION_COLUMNS):
                raise InferenceContractError("Sample submission columns differ from the official schema")
            identifiers = [row.get("ID", "") for row in reader]
    except OSError as error:
        raise InferenceContractError("Cannot read sample submission") from error
    if not identifiers or any(not identifier for identifier in identifiers) or len(identifiers) != len(set(identifiers)):
        raise InferenceContractError("Sample submission IDs must be non-empty and unique")
    return identifiers


def ordered_audio_files(test_dir: Path, identifiers: list[str]) -> list[Path]:
    """Map each requested ID to exactly one supported evaluator file."""
    files = [path for path in test_dir.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_AUDIO_SUFFIXES]
    by_id = {path.stem: path for path in files}
    if len(by_id) != len(files):
        raise InferenceContractError("Evaluator audio IDs are not unique")
    missing = set(identifiers) - set(by_id)
    extra = set(by_id) - set(identifiers)
    if missing or extra:
        raise InferenceContractError(f"Evaluator audio IDs mismatch: missing={sorted(missing)[:3]}, extra={sorted(extra)[:3]}")
    return [by_id[identifier] for identifier in identifiers]


def load_packaged_model(root: Path) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load one local checkpoint, selecting only a declared training mode."""
    checkpoint = root / "model" / "ast"
    try:
        metadata = json.loads((checkpoint / "multitask_config.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise InferenceContractError("Cannot read local AST checkpoint metadata") from error
    mode = metadata.get("mode")
    match mode:
        case "frozen" | "last_block":
            model, processor = load_checkpoint(checkpoint, mode)
        case _:
            raise InferenceContractError("Unsupported local AST checkpoint mode")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return model.eval().to(device), processor


def predict_audio_file(model: ASTMultiTask, processor: ASTFeatureExtractor, path: Path) -> list[float]:
    """Predict one file, rejecting malformed output before it can reach the CSV."""
    probabilities = predict_file(model, processor, load_audio(path)).tolist()
    if len(probabilities) != len(PREDICTION_COLUMNS) or any(
        not math.isfinite(value) or value < 0.0 or value > 1.0 for value in probabilities
    ):
        raise InferenceContractError(f"Invalid probability output for: {path.name}")
    return probabilities


def generate_submission(root: Path) -> Path:
    """Run file-local inference and atomically write evaluator-ordered submission.csv."""
    identifiers = _sample_ids(root / "data" / "sample_submission.csv")
    paths = ordered_audio_files(root / "data" / "test", identifiers)
    model, processor = load_packaged_model(root)
    rows = []
    for index, path in enumerate(paths, 1):
        probabilities = predict_audio_file(model, processor, path)
        rows.append(dict(zip(SUBMISSION_COLUMNS, [path.stem, *probabilities], strict=True)))
        print(f"PREDICT {index}/{len(paths)} {path.stem}", flush=True)
    destination = root / "output" / "submission.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(destination)
    return destination


def main() -> None:
    """Run against an evaluator-layout directory passed as the working directory."""
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    generate_submission(Path.cwd())


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/v2_inference.py


In [11]:
%%writefile /content/dacon236749_v2/package_v2_submission.py
"""Create a minimal self-contained AST-only submission archive."""

from __future__ import annotations

import json
import zipfile
from pathlib import Path
from typing import Final

from package_submission import MAX_EXPANDED_BYTES, MAX_ZIP_BYTES

CHECKPOINT_FILES: Final = (
    "config.json",
    "preprocessor_config.json",
    "multitask_config.json",
    "model.safetensors",
)
RUNTIME_FILES: Final = ("ast_multitask.py", "v2_inference.py")
MODEL_INFO: Final = {
    "base_model": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "revision": "f826b80d28226b62986cc218e5cec390b1096902",
    "license": "BSD-3-Clause",
    "license_source": "https://github.com/YuanGongND/ast/blob/31088be8a3f6ef96416145c4b8d43c81f99eba7a/LICENSE",
    "format": "dacon236749-ast-multitask-v1",
}
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the self-contained v2 AST model with no network access."""
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "model" / "runtime"))

from v2_inference import generate_submission

if __name__ == "__main__":
    generate_submission(ROOT)
'''


def _checkpoint_members(directory: Path) -> dict[str, Path]:
    """Require every inference asset before building an archive or touching the network."""
    members = {name: directory / name for name in CHECKPOINT_FILES}
    missing = [name for name, path in members.items() if not path.is_file()]
    if missing:
        raise ValueError(f"AST checkpoint missing required assets: {', '.join(missing)}")
    try:
        metadata = json.loads(members["multitask_config.json"].read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError("AST checkpoint metadata is not valid JSON") from error
    if metadata.get("format_version") != 1 or metadata.get("mode") not in {"frozen", "last_block"}:
        raise ValueError("AST checkpoint metadata has an unsupported contract")
    return members


def build_ast_submission(checkpoint_dir: Path, output_zip: Path) -> Path:
    """Package only AST weights and local runtime files, without baseline model copies."""
    checkpoint_dir = checkpoint_dir.resolve()
    output_zip = output_zip.resolve()
    members = _checkpoint_members(checkpoint_dir)
    source = Path(__file__).resolve().parent
    runtime = {name: source / name for name in RUNTIME_FILES}
    license_text = (source / "AST_LICENSE.txt").read_bytes()
    missing_runtime = [name for name, path in runtime.items() if not path.is_file()]
    if missing_runtime:
        raise ValueError(f"Submission runtime missing: {', '.join(missing_runtime)}")
    expanded_size = sum(path.stat().st_size for path in members.values())
    expanded_size += sum(path.stat().st_size for path in runtime.values()) + len(DEPLOY_SCRIPT.encode())
    expanded_size += len(json.dumps(MODEL_INFO, sort_keys=True).encode()) + len(license_text)
    if expanded_size > MAX_EXPANDED_BYTES or expanded_size > MAX_ZIP_BYTES:
        raise ValueError(f"AST archive would exceed submission limits: {expanded_size} bytes")
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for name, path in members.items():
                archive.write(path, f"model/ast/{name}")
            archive.writestr("model/ast/MODEL_INFO.json", json.dumps(MODEL_INFO, sort_keys=True))
            archive.writestr("model/ast/LICENSE.txt", license_text)
            for name, path in runtime.items():
                archive.write(path, f"model/runtime/{name}")
            archive.writestr("script.py", DEPLOY_SCRIPT)
            archive.writestr("requirements.txt", "")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            raise ValueError("Built AST archive exceeds compressed submission size limit")
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
    return output_zip


Writing /content/dacon236749_v2/package_v2_submission.py


In [12]:
%%writefile /content/dacon236749_v2/run_v2.py
"""Run dev-only AST selection, then the sealed comparison and offline replay."""

import argparse
import gc
import importlib.metadata
import json
import shutil
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from ast_multitask import load_checkpoint
from graph_features import Extractor, atomic_json, digest, extract_cached
from graph_fusion import graph_predictions
from graph_scoring import SUBMISSION_COLUMNS, official_metrics
from package_submission import build_submission, verify_packaged
from package_v2_submission import build_ast_submission
from run_experiment import read_csv, write_csv
from train_v2 import predict_rows, read_development_manifest, train_candidates
from v2_inference import predict_audio_file
from v2_scoring import select_candidate


def checkpoint_hashes(directory: Path) -> dict[str, str]:
    """Bind selection to every byte needed by the exported inference model."""
    return {p.name: digest(p) for p in sorted(directory.iterdir()) if p.is_file()}


def lock_candidate(path: Path, record: dict) -> None:  # noqa: DICT_OK
    """A completed selection cannot be replaced during an interrupted run."""
    if path.exists():
        if json.loads(path.read_text()) != record:
            message = "A candidate lock already exists with different inputs or weights"
            raise ValueError(message)
        return
    atomic_json(path, record)


def open_locked_test(lock_path: Path, manifest: Path) -> list:
    """Read fresh labels only after verifying the development selection receipt."""
    lock = json.loads(lock_path.read_text())
    if lock["test_manifest_sha256"] != digest(manifest):
        message = "Locked test manifest changed after selection"
        raise ValueError(message)
    rows = read_csv(manifest)
    if not rows or any(r["split"] not in {"test_generator", "test_source"} for r in rows):
        message = "Locked evaluation requires fresh challenge roles only"
        raise ValueError(message)
    return rows


def baseline_predictions(baseline: Path, rows: list, data: Path, out: Path, deadline: float) -> list:
    extractor = Extractor(baseline / "script.py", include_raw=False)
    try:
        features = extract_cached(extractor, [data / r["path"] for r in rows], out / "g0_cache", deadline)
    finally:
        extractor.close()
    if len(features) != len(rows):
        message = "G0 comparison did not finish within the allocated deadline"
        raise TimeoutError(message)
    return [graph_predictions(feature)["G0"] for feature in features]


def compare_fresh(rows: list, predictions: dict) -> dict:  # noqa: DICT_OK
    report = {}
    for split in ("combined", "test_generator", "test_source"):
        chosen = rows if split == "combined" else [r for r in rows if r["split"] == split]
        report[split] = {}
        for name, predicted in predictions.items():
            by_id = {r["ID"]: r for r in predicted}
            report[split][name] = official_metrics(chosen, [by_id[r["ID"]] for r in chosen])
    return report


def package_replay(args, training: dict, selection: dict, deadline: float) -> dict:  # noqa: DICT_OK
    """Exercise fresh scripts on three real clips and a 60-second stress file."""
    data = args.output / "package_inputs"
    (data / "test").mkdir(parents=True, exist_ok=True)
    rows = read_csv(args.data / "development" / "manifest.csv")
    chosen = next(r for r in rows if r["split"] == "dev_select")
    audio, sr = sf.read(args.data / chosen["path"], dtype="float32")
    stress = np.tile(audio, int(np.ceil(60 * sr / len(audio))))[:60 * sr]
    sf.write(data / "test" / "stress_60s.wav", stress, sr)
    for p in sorted((args.baseline / "data" / "test").glob("*.wav")):
        shutil.copy2(p, data / "test" / p.name)
    paths = sorted((data / "test").glob("*.wav"))
    write_csv(data / "sample_submission.csv", [dict(zip(SUBMISSION_COLUMNS, [p.stem, 0, 0, 0, 0, 0])) for p in paths])
    selected = selection["selected"]
    names = ["AST_LAST_BLOCK"] + (["G0"] if selected == "G0" else [])
    if selected not in names:
        names.append(selected)
    receipts = {}
    for name in names:
        runtime_records = []
        destination = args.output / f"{name}_submission.zip"
        if name == "G0":
            build_submission(args.original_zip, destination, {"graph": "G0", "fusion": None})
            test_rows = [{"ID": p.stem, "path": str(p.relative_to(data))} for p in paths]
            reference = baseline_predictions(args.baseline, test_rows, data, args.output, deadline)
            runtime_records = [{k: v for k, v in json.loads(p.read_text()).items() if k != "feature"}
                               for p in (args.output / "g0_cache").glob("*.json")
                               if json.loads(p.read_text())["feature"]["ID"] == "stress_60s"]
        else:
            checkpoint = Path(training["candidates"][name]["checkpoint"])
            build_ast_submission(checkpoint, destination)
            mode = json.loads((checkpoint / "multitask_config.json").read_text())["mode"]
            model, processor = load_checkpoint(checkpoint, mode)
            model = model.to("cuda").eval()
            reference = []
            for p in paths:
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                started = time.perf_counter()
                probability = predict_audio_file(model, processor, p)
                torch.cuda.synchronize()
                reference.append(dict(zip(SUBMISSION_COLUMNS, [p.stem, *probability])))
                runtime_records.append({"ID": p.stem, "seconds": time.perf_counter() - started,
                    "max_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
                    "max_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3})
            del model, processor
            gc.collect()
            torch.cuda.empty_cache()
        fresh = args.work / f"unpacked_{name}"
        with zipfile.ZipFile(destination) as archive:
            archive.extractall(fresh)
        shutil.copytree(data, fresh / "data", dirs_exist_ok=True)
        start = time.perf_counter()
        packaged = verify_packaged(fresh, deadline)
        delta = max(abs(float(a[c]) - float(b[c])) for a, b in zip(reference, packaged, strict=True)
                    for c in SUBMISSION_COLUMNS[1:])
        if [r["ID"] for r in reference] != [r["ID"] for r in packaged] or delta > 1e-5:
            message = f"Fresh offline package parity failed: {name}, delta={delta}"
            raise AssertionError(message)
        shutil.copy2(fresh / "output" / "packaged_inference.log", args.output / f"{name}_package.log")
        write_csv(args.output / f"{name}_package_reference.csv", reference)
        receipts[name] = {"zip": str(destination), "zip_sha256": digest(destination),
            "max_probability_delta": delta, "script_seconds_including_model_load": time.perf_counter() - start,
            "stress_60s_included": True, "file_count": len(paths), "gpu": torch.cuda.get_device_name(),
            "reference_runtime": runtime_records, "L4_60_minute_compliance": "unverified"}
        shutil.rmtree(fresh)
        atomic_json(args.output / "packaged_verification.json", receipts)
    return receipts


def main() -> None:
    parser = argparse.ArgumentParser()
    for name in ("assets", "data", "baseline", "original-zip", "output", "work"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    args = parser.parse_args()
    args.output.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if start >= args.deadline - 1200:
        message = "Insufficient GPU budget to begin the comparison"
        raise TimeoutError(message)
    atomic_json(args.output / "environment.json", {"started_unix": start, "python": sys.version,
        "gpu": torch.cuda.get_device_name(), "gpu_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3,
        "packages": {k: importlib.metadata.version(k) for k in
                     ("torch", "torchaudio", "transformers", "numpy", "scikit-learn", "soundfile")}})
    development = args.data / "development" / "manifest.csv"
    rows = read_development_manifest(development, args.data)
    dev_select = [r for r in rows if r["split"] == "dev_select"]
    g0 = baseline_predictions(args.baseline, dev_select, args.data, args.output, args.deadline - 1200)
    write_csv(args.output / "G0_dev_predictions.csv", g0)
    training = train_candidates(args.assets, development, args.data, args.output / "training", args.deadline)
    if any("predictions" not in value for value in training["candidates"].values()):
        message = "The allocated budget did not permit all declared candidates to finish"
        raise TimeoutError(message)
    predictions = {"G0": g0, **{name: value["predictions"] for name, value in training["candidates"].items()}}
    selection = select_candidate(dev_select, predictions)
    atomic_json(args.output / "selection.json", selection)
    for name, value in predictions.items():
        write_csv(args.output / f"{name}_dev_predictions.csv", value)
    manifest = args.data / "locked_test" / "manifest.csv"
    lock = {"selected": selection["selected"], "development_manifest_sha256": digest(development),
        "test_manifest_sha256": digest(manifest), "original_zip_sha256": digest(args.original_zip),
        "checkpoints": {name: checkpoint_hashes(Path(value["checkpoint"])) for name, value in training["candidates"].items()},
        "selection_sha256": digest(args.output / "selection.json")}
    lock_candidate(args.output / "candidate_lock.json", lock)
    fresh_rows = open_locked_test(args.output / "candidate_lock.json", manifest)
    atomic_json(args.output / "test_opened.json", {"unix": time.time(), "candidate_lock_sha256": digest(args.output / "candidate_lock.json")})
    fresh = {"G0": baseline_predictions(args.baseline, fresh_rows, args.data, args.output, args.deadline - 600)}
    write_csv(args.output / "G0_test_predictions.csv", fresh["G0"])
    for name, value in training["candidates"].items():
        checkpoint = Path(value["checkpoint"])
        mode = json.loads((checkpoint / "multitask_config.json").read_text())["mode"]
        model, processor = load_checkpoint(checkpoint, mode)
        model = model.to("cuda").eval()
        fresh[name] = predict_rows(model, processor, fresh_rows, args.data, torch.device("cuda"), deadline=args.deadline - 600)
        write_csv(args.output / f"{name}_test_predictions.csv", fresh[name])
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
    comparison = {"selection": selection, "fresh_test": compare_fresh(fresh_rows, fresh),
        "scope": "external_source_disjoint_validation_not_DACON_leaderboard", "fresh_clips": len(fresh_rows)}
    atomic_json(args.output / "comparison.json", comparison)
    print("COMPARISON_FINISHED", json.dumps(comparison, ensure_ascii=False), flush=True)
    package_replay(args, training, selection, args.deadline - 120)
    atomic_json(args.output / "finished.json", {"status": "finished", "elapsed_seconds": time.time() - start,
        "selected": selection["selected"], "submission_status": "not_submitted_in_this_run"})
    print("V2_FINISHED", args.output, flush=True)


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/run_v2.py


In [13]:
%%writefile /content/dacon236749_v2/v2_data.py
"""Small shared data/audio primitives for the V2 bank."""
from __future__ import annotations

import hashlib
import bz2
import subprocess
import struct
import wave
import zlib
from collections.abc import Callable, Sequence
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final, TypeVar, assert_never

import imageio_ffmpeg
import numpy as np

LABEL_NAMES: Final = ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT")
Candidate = TypeVar("Candidate")
Loaded = TypeVar("Loaded")


class DataIntegrityError(RuntimeError):
    """A retained data item failed an integrity boundary."""


@dataclass(frozen=True, slots=True)
class Labels:
    FILE_FAKE: str
    VOICE_FAKE: str
    MUSIC_FAKE: str
    VOICE_PRESENT: str
    MUSIC_PRESENT: str


@dataclass(frozen=True, slots=True)
class SourceRow:
    id: str
    prepared_path: str
    original_path: str
    source_kind: str
    group_id: str
    provider: str
    content_id: str
    source_url: str
    revision: str
    source_sha256: str
    pcm_sha256: str
    license: str
    attribution: str
    labels: Labels

    def json_record(self) -> dict[str, str | dict[str, str]]:
        return asdict(self)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def decode_audio(path: Path, limit_seconds: int = 10) -> np.ndarray:
    result = subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-v", "error", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
        check=True, capture_output=True,
    )
    audio = np.frombuffer(result.stdout, dtype="<f4").copy()
    if not np.isfinite(audio).all():
        raise ArithmeticError(f"non-finite decoded audio: {path}")
    limit = limit_seconds * 16000
    start = max(0, (len(audio) - limit) // 2)
    return audio[start:start + limit]


def pcm_sha256(audio: np.ndarray) -> str:
    normalized = np.asarray(audio, dtype="<f4")
    return hashlib.sha256(normalized.tobytes()).hexdigest()


def write_wav(path: Path, audio: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
    pcm = np.round(audio / max(1.0, peak / 0.98) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as handle:
        handle.setparams((1, 2, 16000, 0, "NONE", "not compressed"))
        handle.writeframes(pcm.tobytes())


def stable_key(value: str) -> str:
    return hashlib.sha256(value.encode()).hexdigest()


def first_valid(candidates: Sequence[Candidate], loader: Callable[[Candidate], Loaded | None], count: int) -> list[Loaded]:
    accepted: list[Loaded] = []
    for start in range(0, len(candidates), 8):
        with ThreadPoolExecutor(max_workers=8) as executor:
            accepted.extend(item for item in executor.map(loader, candidates[start:start + 8]) if item is not None)
        if len(accepted) >= count:
            return accepted[:count]
    return accepted


def extract_local_member(blob: bytes, entry: dict[str, int | str], output: Path) -> Path:
    if len(blob) < 30:
        raise DataIntegrityError("ZIP member local header is truncated")
    signature, _, _, method, _, _, _, _, _, name_len, extra_len = struct.unpack("<4s5H3L2H", blob[:30])
    if signature != b"PK\x03\x04" or method != int(entry["compress_type"]):
        raise DataIntegrityError("ZIP member header or compression method mismatch")
    start = 30 + name_len + extra_len
    compressed = blob[start:start + int(entry["compress_size"])]
    if method not in (0, 8, 12):
        raise DataIntegrityError(f"unsupported ZIP compression method {method}")
    match method:
        case 0:
            payload = compressed
        case 8:
            payload = zlib.decompress(compressed, -15)
        case 12:
            payload = bz2.decompress(compressed)
        case unreachable:
            assert_never(unreachable)
    if len(payload) != int(entry["file_size"]):
        raise DataIntegrityError("ZIP member size mismatch")
    if zlib.crc32(payload) != int(entry["CRC"]):
        raise DataIntegrityError("ZIP member CRC mismatch")
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_bytes(payload)
    return output


Writing /content/dacon236749_v2/v2_data.py


In [14]:
%%writefile /content/dacon236749_v2/prepare_v2_sources.py
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.11"
# dependencies = ["numpy==2.4.6", "pyarrow==21.0.0", "imageio-ffmpeg==0.6.0"]
# ///
# How to run: `uv run prepare_v2_sources.py --output _workspace/v2_sources`.
"""Acquire the exact licensed audio retained for the V2 experiment."""
from __future__ import annotations

import argparse
import csv
import json
import shutil
import subprocess
import unicodedata
import urllib.parse
from pathlib import Path
from collections.abc import Sequence
from typing import Final

import numpy as np
import pyarrow.parquet as pq

from v2_data import DataIntegrityError as SourceIntegrityError
from v2_data import Labels, SourceRow, decode_audio, extract_local_member, first_valid, pcm_sha256, sha256_file, stable_key, write_wav

ROOT: Final = Path(__file__).resolve().parent
MLAAD_REV: Final = "9143e5ea709575ebab6bec52840a1043aada7bb1"
ECHOES_REV: Final = "14b0c76c6a691c42fadfab9fb6a4eb1ee8c628a2"
NSYNTH_REV: Final = "5ba57c8114d9d4d7cf076c4cb3c8f508f105fca8"
MLAAD_GENERATORS: Final = (
    "Cartesia.ai (Sonic-3)", "Higgs-Audio-V2", "MeloTTS", "FireRedTTS-2.0",
    "Index-TTS-1.5", "Index-TTS-2.0", "Llasa-1B-Multilingual", "Mars5",
    "OpenVoiceV2", "VoxCPM-0.5B",
)
MLAAD_FALLBACKS: Final = ("Llasa-8B", "RVC")
DEV_MUSIC_PROVIDERS: Final = ("acestep", "musicgen", "audioldm")
TEST_MUSIC_PROVIDERS: Final = ("suno", "udio", "elevenlabs")
def real_voice_labels() -> Labels:
    return Labels("0", "0", "", "1", "0")


def fake_voice_labels() -> Labels:
    return Labels("1", "1", "", "1", "0")


def fetch(url: str, path: Path) -> Path:
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_suffix(path.suffix + ".part")
    subprocess.run(["curl", "-fLsS", "--retry", "3", "--max-time", "180", url, "-o", str(part)], check=True)
    part.replace(path)
    return path


def validate_inventory(rows: list[SourceRow], old_groups: set[str], old_pcm: set[str]) -> None:
    groups = [part for row in rows for part in row.group_id.split("|")]
    repeated_pcm = len({row.pcm_sha256 for row in rows}) != len(rows)
    if set(groups) & old_groups or {row.pcm_sha256 for row in rows} & old_pcm or repeated_pcm:
        raise SourceIntegrityError("source group or decoded PCM overlaps another retained source")
    if any(not row.license or not row.source_sha256 for row in rows):
        raise SourceIntegrityError("every retained source requires a license and checksum")
    known = {"mlaad_real", "mlaad_fake", "fma_real", "echoes_tta", "nsynth"}
    if {row.source_kind for row in rows} - known:
        raise SourceIntegrityError("unknown source kind")


def distinct_artists(items: Sequence[dict[str, str]], excluded: set[str], count: int) -> list[dict[str, str]]:
    selected: list[dict[str, str]] = []
    artists = set(excluded)
    for item in items:
        artist = item["artist_id"]
        if artist in artists:
            continue
        selected.append(item)
        artists.add(artist)
        if len(selected) == count:
            break
    return selected


def _normalized(value: str) -> str:
    plain = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode().lower()
    return "".join(char for char in plain if char.isalnum())


def _retain(output: Path, original: Path, kind: str, group: str, provider: str, content: str, url: str, revision: str, license_name: str, attribution: str, labels: Labels) -> SourceRow:
    audio = decode_audio(original)
    if len(audio) < 64000:
        raise SourceIntegrityError(f"audio shorter than four seconds: {original}")
    identifier = stable_key(f"{kind}|{provider}|{content}")[:20]
    prepared = output / "prepared" / f"{identifier}.wav"
    write_wav(prepared, audio)
    return SourceRow(identifier, str(prepared.relative_to(output)), str(original.relative_to(output)), kind, group, provider, content, url, revision, sha256_file(original), pcm_sha256(decode_audio(prepared)), license_name, attribution, labels)


def acquire_speech(output: Path, old_groups: set[str]) -> tuple[list[SourceRow], list[str]]:
    base = f"https://huggingface.co/datasets/mueller91/MLAAD-tiny/resolve/{MLAAD_REV}/"
    api = f"https://huggingface.co/api/datasets/mueller91/MLAAD-tiny/tree/{MLAAD_REV}/"
    rows: list[SourceRow] = []
    used_groups: set[str] = set()
    accepted_generators: list[str] = []
    for generator in (*MLAAD_GENERATORS, *MLAAD_FALLBACKS):
        listing_path = fetch(api + urllib.parse.quote(f"fake/en/{generator}", safe="/") + "?limit=1000", output / "metadata" / f"mlaad_{stable_key(generator)[:12]}.json")
        entries = [item for item in json.loads(listing_path.read_text()) if item["type"] == "file"]
        entries.sort(key=lambda item: stable_key(item["path"]))
        candidates = []
        for entry in entries:
            filename = Path(entry["path"]).name
            group = "mlaad:" + Path(filename).stem
            if group in old_groups or group in used_groups:
                continue
            candidates.append((entry, filename, group))

        def load_pair(candidate: tuple[dict[str, str | int], str, str]) -> tuple[str, tuple[SourceRow, SourceRow]] | None:
            entry, filename, group = candidate
            fake_url = base + urllib.parse.quote(entry["path"], safe="/")
            real_name = f"original/en/{filename}"
            real_url = base + urllib.parse.quote(real_name, safe="/")
            fake_path = output / "originals" / entry["path"]
            real_path = output / "originals" / real_name
            try:
                fetch(fake_url, fake_path)
                fetch(real_url, real_path)
                pair = (
                    _retain(output, real_path, "mlaad_real", group, "M-AILABS", filename, real_url, MLAAD_REV, "M-AILABS permissive notice", filename, real_voice_labels()),
                    _retain(output, fake_path, "mlaad_fake", group, generator, filename, fake_url, MLAAD_REV, "CC-BY-NC-4.0", generator, fake_voice_labels()),
                )
            except (subprocess.CalledProcessError, SourceIntegrityError):
                return None
            return group, pair

        accepted_pairs = first_valid(candidates, load_pair, 20)
        accepted = len(accepted_pairs)
        if accepted != 20:
            continue
        for group, pair in accepted_pairs:
            rows.extend(pair)
            used_groups.add(group)
        accepted_generators.append(generator)
        if len(accepted_generators) == 10:
            break
    if len(accepted_generators) != 10:
        raise SourceIntegrityError(f"only {len(accepted_generators)} speech generators yielded 20 valid pairs")
    for name in ("LICENSE", "original/LICENSE", "README.md"):
        fetch(base + name, output / "licenses" / ("MLAAD_" + name.replace("/", "_")))
    return rows, accepted_generators


def _echoes_member(entry: dict[str, int | str], output: Path) -> Path:
    if output.exists():
        return output
    start = int(entry["header_offset"])
    end = start + 30 + len(str(entry["filename"]).encode()) + int(entry["compress_size"]) + 512
    url = f"https://huggingface.co/datasets/Octavian97/Echoes/resolve/{ECHOES_REV}/Echoes.zip?member={stable_key(str(entry['filename']))[:12]}"
    result = subprocess.run(["curl", "-fLsS", "--retry", "3", "--max-time", "180", "--range", f"{start}-{end}", url], check=True, capture_output=True)
    return extract_local_member(result.stdout, entry, output)


def acquire_music(output: Path) -> list[SourceRow]:
    candidates = json.loads((ROOT / "_workspace/v2_discovery/echoes_fma_unique.json").read_text())
    metadata = list(csv.DictReader((ROOT / "_workspace/v2_discovery/archive_probes/echoes_manifest.csv").open()))
    entries = {item["filename"]: item for item in json.loads((ROOT / "_workspace/v2_discovery/archive_probes/echoes_entries.json").read_text())}
    generated = {(_normalized(row["original_audio"]), row["generator"]): row for row in metadata if row["type"] == "TTA"}
    ordered = sorted(candidates, key=lambda item: stable_key(item["group_id"] + "|" + item["echoes_original_audio"]))
    test_candidates = distinct_artists([item for item in ordered if all((_normalized(item["echoes_original_audio"]), p) in generated for p in TEST_MUSIC_PROVIDERS)], set(), 68)

    def load_reference(pair: tuple[dict[str, str], tuple[str, str, str]]) -> list[SourceRow] | None:
        item, providers = pair
        group, content = item["group_id"], item["echoes_original_audio"]
        try:
            fma_url = "https://files.freemusicarchive.org/storage-freemusicarchive-org/" + urllib.parse.quote(item["track_file"], safe="/")
            fma_path = fetch(fma_url, output / "originals" / "fma" / f"{item['track_id']}.mp3")
            retained = [_retain(output, fma_path, "fma_real", group, "FMA", content, fma_url, "FMA metadata snapshot", item["license_title"] + " " + item["license_url"], item["artist_name"], Labels("0", "", "0", "", "1"))]
            for provider in providers:
                meta = generated[(_normalized(content), provider)]
                archive_name = "Echoes/" + meta["path_in_dataset"]
                original = output / "originals" / "echoes" / meta["path_in_dataset"]
                _echoes_member(entries[archive_name], original)
                retained.append(_retain(output, original, "echoes_tta", group, provider, content, f"Echoes.zip::{archive_name}", ECHOES_REV, "CC-BY-SA-4.0", f"Echoes; {provider}", Labels("1", "", "1", "", "1")))
        except (subprocess.CalledProcessError, SourceIntegrityError, KeyError):
            return None
        return retained

    test_loaded = first_valid([(item, TEST_MUSIC_PROVIDERS) for item in test_candidates], load_reference, 20)
    if len(test_loaded) != 20:
        raise SourceIntegrityError("fewer than 20 test-provider FMA artists could be retained")
    held_artists = {rows[0].group_id for rows in test_loaded}
    dev_candidates = distinct_artists([item for item in ordered if all((_normalized(item["echoes_original_audio"]), p) in generated for p in DEV_MUSIC_PROVIDERS)], {group.removeprefix("fma_artist_") for group in held_artists}, 106)
    dev_loaded = first_valid([(item, DEV_MUSIC_PROVIDERS) for item in dev_candidates], load_reference, 40)
    if len(dev_loaded) != 40:
        raise SourceIntegrityError("fewer than 40 development-provider FMA artists could be retained")
    return [row for group_rows in dev_loaded + test_loaded for row in group_rows]


def acquire_nsynth(output: Path) -> list[SourceRow]:
    source = ROOT / "_workspace/v2_discovery/nsynth_test.parquet"
    preserved = output / "originals/nsynth/test.parquet"
    preserved.parent.mkdir(parents=True, exist_ok=True)
    if not preserved.exists():
        shutil.copyfile(source, preserved)
    parquet = pq.ParquetFile(source)
    metadata = []
    for batch in parquet.iter_batches(batch_size=256, columns=["id", "instrument", "instrument_str", "instrument_family_str", "instrument_source_str"]):
        metadata.extend(batch.to_pylist())
    eligible = [item for item in metadata if item["instrument_source_str"] == "acoustic" and item["instrument_family_str"] != "vocal"]
    by_instrument: dict[int, list[dict[str, str | int]]] = {}
    for item in eligible:
        by_instrument.setdefault(int(item["instrument"]), []).append(item)
    eligible_ids = {str(item["id"]) for item in eligible}
    chosen: dict[int, dict[str, tuple[str, dict[str, str | int | dict[str, list[float] | str | int]], np.ndarray]]] = {}
    for batch in parquet.iter_batches(batch_size=64, columns=["id", "instrument", "instrument_str", "audio"]):
        for item in batch.to_pylist():
            if item["id"] not in eligible_ids:
                continue
            instrument = int(item["instrument"])
            audio = np.asarray(item["audio"]["array"], dtype=np.float32)
            digest = pcm_sha256(audio)
            options = chosen.setdefault(instrument, {})
            candidate = (stable_key(str(item["id"])), item, audio)
            if digest not in options or candidate[0] < options[digest][0]:
                options[digest] = candidate
            if len(options) > 4:
                del options[max(options, key=lambda key: options[key][0])]
    if len(chosen) != 25 or any(len(items) != 4 for items in chosen.values()):
        raise SourceIntegrityError("NSynth acoustic non-vocal selection must be 100 notes from 25 instruments")
    rows: list[SourceRow] = []
    for instrument in sorted(chosen):
        for _, item, audio in sorted(chosen[instrument].values(), key=lambda value: value[0]):
            identifier = stable_key("nsynth|" + item["id"])[:20]
            original = output / "originals/nsynth/selected" / f"{identifier}.wav"
            prepared = output / "prepared" / f"{identifier}.wav"
            write_wav(original, audio)
            write_wav(prepared, audio)
            rows.append(SourceRow(identifier, str(prepared.relative_to(output)), str(original.relative_to(output)), "nsynth", f"nsynth_instrument_{item['instrument']}", "NSynth", item["id"], "NSynth test.parquet row", NSYNTH_REV, sha256_file(original), pcm_sha256(decode_audio(prepared)), "CC-BY-4.0", item["instrument_str"], Labels("0", "", "0", "0", "1")))
    return rows


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", type=Path, default=ROOT / "_workspace/v2_sources")
    parser.add_argument("--report", type=Path)
    args = parser.parse_args()
    output: Path = args.output.resolve()
    old_rows = list(csv.DictReader((ROOT / "validation/manifest.csv").open()))
    old_groups = {part for row in old_rows for part in row["group_id"].split("|")}
    old_pcm = {pcm_sha256(decode_audio(ROOT / "validation" / row["path"])) for row in old_rows}
    speech_rows, accepted_generators = acquire_speech(output, old_groups)
    rows = speech_rows + acquire_music(output) + acquire_nsynth(output)
    validate_inventory(rows, old_groups, old_pcm)
    artifacts = [ROOT / "_workspace/v2_discovery/nsynth_test.parquet", ROOT / "_workspace/v2_discovery/archive_probes/echoes_manifest.csv", ROOT / "_workspace/v2_discovery/archive_probes/echoes_entries.json", ROOT / "_workspace/v2_discovery/echoes_fma_unique.json"]
    inventory = {"schema_version": 1, "sources": [row.json_record() for row in rows], "source_artifacts": [{"path": str(path.relative_to(ROOT)), "sha256": sha256_file(path), "bytes": path.stat().st_size} for path in artifacts], "acquisition": {"mlaad_revision": MLAAD_REV, "accepted_voice_generators": accepted_generators, "test_voice_generators": accepted_generators[-2:], "echoes_revision": ECHOES_REV, "nsynth_revision": NSYNTH_REV, "aime_rows": 0}}
    output.mkdir(parents=True, exist_ok=True)
    inventory_path = output / "source_inventory.json"
    inventory_path.write_text(json.dumps(inventory, indent=2, ensure_ascii=False) + "\n")
    report = {"sources": len(rows), "speech": sum(row.source_kind.startswith("mlaad") for row in rows), "music": sum(row.source_kind in {"fma_real", "echoes_tta"} for row in rows), "nsynth": sum(row.source_kind == "nsynth" for row in rows), "groups": len({row.group_id for row in rows}), "inventory_sha256": sha256_file(inventory_path)}
    if args.report:
        args.report.parent.mkdir(parents=True, exist_ok=True)
        args.report.write_text(json.dumps(report, indent=2) + "\n")
    print(json.dumps(report))


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/prepare_v2_sources.py


In [15]:
%%writefile /content/dacon236749_v2/prepare_v2_validation.py
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.11"
# dependencies = ["numpy==2.4.6", "imageio-ffmpeg==0.6.0"]
# ///
# How to run: `uv run prepare_v2_validation.py --sources _workspace/v2_sources/source_inventory.json`.
"""Build source-disjoint development and sealed challenge manifests."""
from __future__ import annotations

import argparse
import csv
import json
import shutil
import zipfile
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Final

import numpy as np

from v2_data import Labels, decode_audio, pcm_sha256, sha256_file, stable_key, write_wav

ROOT: Final = Path(__file__).resolve().parent
FIELDS: Final = ("ID", "path", "group_id", "split", "source", "FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT")
ROLES: Final = ("dev_train", "dev_select", "test_generator", "test_source")


class SplitIntegrityError(RuntimeError):
    """A data row violates the sealed split contract."""


@dataclass(frozen=True, slots=True)
class ManifestRow:
    ID: str
    path: str
    group_id: str
    split: str
    source: str
    FILE_FAKE: str
    VOICE_FAKE: str
    MUSIC_FAKE: str
    VOICE_PRESENT: str
    MUSIC_PRESENT: str
    pcm_sha256: str

    def csv_record(self) -> dict[str, str]:
        data = asdict(self)
        data.pop("pcm_sha256")
        return data


def write_manifest(path: Path, rows: list[ManifestRow]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(row.csv_record() for row in rows)


def read_manifest(path: Path) -> list[ManifestRow]:
    with path.open() as handle:
        return [ManifestRow(**row, pcm_sha256="") for row in csv.DictReader(handle)]


def audit_rows(rows: list[ManifestRow]) -> None:
    roles: dict[str, set[str]] = {}
    for row in rows:
        if row.split not in ROLES:
            raise SplitIntegrityError(f"unknown role: {row.split}")
        for group in row.group_id.split("|"):
            roles.setdefault(group, set()).add(row.split)
        values = (row.FILE_FAKE, row.VOICE_FAKE, row.MUSIC_FAKE, row.VOICE_PRESENT, row.MUSIC_PRESENT)
        if any(value not in {"", "0", "1"} for value in values):
            raise SplitIntegrityError("labels must be 0, 1, or explicit unknown")
    if any(len(value) != 1 for value in roles.values()):
        raise SplitIntegrityError("atomic source group crosses roles")
    hashes = [row.pcm_sha256 for row in rows]
    if len(hashes) != len(set(hashes)):
        raise SplitIntegrityError("decoded PCM is duplicated")


def _old_rows(old: Path) -> list[ManifestRow]:
    raw = list(csv.DictReader((old / "manifest.csv").open()))
    parent: dict[str, str] = {}

    def find(item: str) -> str:
        parent.setdefault(item, item)
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    def union(left: str, right: str) -> None:
        left, right = find(left), find(right)
        if left != right:
            parent[right] = left

    for row in raw:
        groups = row["group_id"].split("|")
        for group in groups[1:]:
            union(groups[0], group)
        find(groups[0])
    components: dict[str, list[str]] = {}
    for group in parent:
        components.setdefault(find(group), []).append(group)
    role_by_group = {}
    for groups in components.values():
        role = "dev_select" if int(stable_key("|".join(sorted(groups)))[:8], 16) % 5 == 0 else "dev_train"
        role_by_group.update({group: role for group in groups})
    result = []
    for row in raw:
        source = old / row["path"]
        voice_fake = "" if row["source"].startswith("mixture:") and row["VOICE_FAKE"] == "0" and row["MUSIC_FAKE"] == "1" else row["VOICE_FAKE"]
        result.append(ManifestRow(row["ID"], f"audio/{row['ID']}.wav", row["group_id"], role_by_group[row["group_id"].split("|")[0]], row["source"], row["FILE_FAKE"], voice_fake, row["MUSIC_FAKE"], row["VOICE_PRESENT"], row["MUSIC_PRESENT"], pcm_sha256(decode_audio(source))))
    return result


def _role_for_source(item: dict[str, str | dict[str, str]], test_voice_generators: set[str]) -> str:
    provider, kind, group = str(item["provider"]), str(item["source_kind"]), str(item["group_id"])
    if provider in test_voice_generators | {"suno", "udio", "elevenlabs"}:
        return "test_generator"
    bucket = int(stable_key(group)[:8], 16) % 20
    if bucket < 3:
        return "test_source"
    if bucket < 6:
        return "dev_select"
    return "dev_train"


def _select_new(items: list[dict[str, str | dict[str, str]]], test_voice_generators: set[str]) -> list[tuple[dict[str, str | dict[str, str]], str]]:
    music_test_groups = {str(item["group_id"]) for item in items if item["provider"] in {"suno", "udio", "elevenlabs"}}
    voice_test_groups = {str(item["group_id"]) for item in items if item["provider"] in test_voice_generators}
    generator_test_groups = music_test_groups | voice_test_groups
    categorized: dict[str, list[dict[str, str | dict[str, str]]]] = {role: [] for role in ROLES}
    for item in items:
        kind, group = str(item["source_kind"]), str(item["group_id"])
        role = "test_generator" if group in generator_test_groups else _role_for_source(item, test_voice_generators)
        categorized[role].append(item)
    selected: list[tuple[dict[str, str | dict[str, str]], str]] = []
    for item in items:
        role = "test_generator" if str(item["group_id"]) in generator_test_groups else _role_for_source(item, test_voice_generators)
        selected.append((item, role))
    return selected


def _copy_rows(selected: list[tuple[dict[str, str | dict[str, str]], str]], sources: Path, output: Path) -> list[ManifestRow]:
    rows = []
    for item, role in selected:
        source = sources.parent / str(item["prepared_path"])
        identifier = str(item["id"])
        destination = output / "audio" / f"{identifier}.wav"
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
        labels = item["labels"]
        if not isinstance(labels, dict):
            raise SplitIntegrityError("inventory labels must be an object")
        rows.append(ManifestRow(identifier, f"audio/{identifier}.wav", str(item["group_id"]), role, f"{item['source_kind']}/{item['provider']}", *(str(labels[name]) for name in FIELDS[5:]), str(item["pcm_sha256"])))
    return rows


def pure_voice_groups(rows: list[ManifestRow], role: str) -> list[str]:
    return sorted({row.group_id for row in rows if row.split == role and row.VOICE_PRESENT == "1" and row.MUSIC_PRESENT == "0"}, key=stable_key)


def paired_music_groups(rows: list[ManifestRow], role: str) -> list[str]:
    groups = {row.group_id for row in rows if row.split == role and row.MUSIC_PRESENT == "1" and row.VOICE_PRESENT == ""}
    return sorted((group for group in groups if {row.MUSIC_FAKE for row in rows if row.group_id == group} == {"0", "1"}), key=stable_key)


def mixture_voice_label(voice_fake: str, music_fake: str) -> str:
    return "" if voice_fake == "0" and music_fake == "1" else voice_fake


def _mixtures(rows: list[ManifestRow], output: Path) -> list[ManifestRow]:
    result = []
    for role in ROLES:
        voice_groups = pure_voice_groups(rows, role)[:2]
        music_groups = paired_music_groups(rows, role)[:2]
        if len(voice_groups) < 2 or len(music_groups) < 2:
            raise SplitIntegrityError(f"mixture bank requires two voice/music groups in {role}")
        for voice_group, music_group in zip(voice_groups, music_groups, strict=True):
            for voice_fake in ("0", "1"):
                voice = next(row for row in rows if row.group_id == voice_group and row.VOICE_FAKE == voice_fake)
                for music_fake in ("0", "1"):
                    music = next(row for row in rows if row.group_id == music_group and row.MUSIC_FAKE == music_fake)
                    foreground, background = decode_audio(output / voice.path), decode_audio(output / music.path)
                    length = min(len(foreground), len(background))
                    foreground, background = foreground[:length], background[:length]
                    for snr in (0, 10):
                        gain = np.sqrt(np.mean(foreground ** 2) / max(float(np.mean(background ** 2)), 1e-12)) / 10 ** (snr / 20)
                        audio = foreground + gain * background
                        identifier = stable_key(f"mix|{voice.ID}|{music.ID}|{snr}")[:20]
                        path = output / "audio" / f"{identifier}.wav"
                        write_wav(path, audio)
                        result.append(ManifestRow(identifier, f"audio/{identifier}.wav", voice_group + "|" + music_group, role, f"mixture/SNR={snr}", str(max(int(voice_fake), int(music_fake))), mixture_voice_label(voice_fake, music_fake), music_fake, "1", "1", pcm_sha256(decode_audio(path))))
    return result


def _label_audit(rows: list[ManifestRow]) -> dict[str, dict[str, list[str] | int]]:
    report = {}
    for name in FIELDS[5:]:
        values = [getattr(row, name) for row in rows]
        report[name] = {"classes": sorted(set(values) - {""}), "unknown": values.count("")}
    return report


def connected_count(rows: list[ManifestRow], role: str) -> int:
    parent: dict[str, str] = {}

    def find(item: str) -> str:
        parent.setdefault(item, item)
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    for row in rows:
        if row.split != role:
            continue
        groups = row.group_id.split("|")
        root = find(groups[0])
        for group in groups[1:]:
            parent[find(group)] = root
    return len({find(group) for group in parent})


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--sources", type=Path, required=True)
    parser.add_argument("--old", type=Path, default=ROOT / "validation")
    parser.add_argument("--output", type=Path, default=ROOT / "v2_validation")
    parser.add_argument("--zip", dest="zip_path", type=Path, default=ROOT / "_workspace/v2_validation.zip")
    parser.add_argument("--audit", type=Path)
    args = parser.parse_args()
    inventory = json.loads(args.sources.read_text())
    output: Path = args.output.resolve()
    old_rows = _old_rows(args.old.resolve())
    copied_old = []
    for row in old_rows:
        destination = output / row.path
        destination.parent.mkdir(parents=True, exist_ok=True)
        write_wav(destination, decode_audio(args.old.resolve() / "audio" / f"{row.ID}.wav"))
        copied_old.append(replace(row, pcm_sha256=pcm_sha256(decode_audio(destination))))
    test_voice_generators = set(inventory["acquisition"]["test_voice_generators"])
    rows = copied_old + _copy_rows(_select_new(inventory["sources"], test_voice_generators), args.sources.resolve(), output)
    rows += _mixtures(rows, output)
    audit_rows(rows)
    used_paths = {row.path for row in rows}
    for path in (output / "audio").glob("*"):
        if str(path.relative_to(output)) not in used_paths:
            path.unlink()
    development = sorted([row for row in rows if row.split.startswith("dev_")], key=lambda row: row.ID)
    locked = sorted([row for row in rows if row.split.startswith("test_")], key=lambda row: row.ID)
    if not 400 <= len(rows) <= 1100:
        raise SplitIntegrityError(f"final bank has {len(rows)} rows outside [400, 1100]")
    if any(connected_count(locked, role) < 20 for role in ("test_generator", "test_source")):
        raise SplitIntegrityError("each challenge requires at least 20 connected lineage components")
    labels = _label_audit(locked)
    if any(value["classes"] != ["0", "1"] for value in labels.values()):
        raise SplitIntegrityError("combined locked challenge lacks both classes for an official target")
    development_path = output / "development/manifest.csv"
    locked_path = output / "locked_test/manifest.csv"
    write_manifest(development_path, development)
    write_manifest(locked_path, locked)
    pcm_path = output / "pcm_sha256.csv"
    with pcm_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=("ID", "pcm_sha256"))
        writer.writeheader()
        writer.writerows({"ID": row.ID, "pcm_sha256": row.pcm_sha256} for row in sorted(rows, key=lambda item: item.ID))
    audit = {"rows": len(rows), "mixtures": sum(row.source.startswith("mixture/") for row in rows), "role_counts": {role: sum(row.split == role for row in rows) for role in ROLES}, "atomic_groups": {role: len({part for row in rows if row.split == role for part in row.group_id.split("|")}) for role in ROLES}, "connected_components": {role: connected_count(rows, role) for role in ROLES}, "voice_authenticity_masked_for_generated_music": sum(row.VOICE_FAKE == "" and row.MUSIC_FAKE == "1" and row.VOICE_PRESENT == "1" for row in rows), "locked_label_coverage": labels, "development_manifest_sha256": sha256_file(development_path), "locked_manifest_sha256": sha256_file(locked_path), "pcm_index_sha256": sha256_file(pcm_path)}
    (output / "split_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    if args.audit:
        args.audit.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(output / "split_audit.json", args.audit)
    args.zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(args.zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob("*")):
            if path.is_file():
                info = zipfile.ZipInfo("v2_validation/" + str(path.relative_to(output)), (2026, 9, 13, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                archive.writestr(info, path.read_bytes())
        info = zipfile.ZipInfo("v2_validation/DATA_SOURCES_V2.md", (2026, 9, 13, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, (ROOT / "DATA_SOURCES_V2.md").read_bytes())
    print(json.dumps({**audit, "zip": str(args.zip_path), "zip_sha256": sha256_file(args.zip_path)}))


if __name__ == "__main__":
    main()


Writing /content/dacon236749_v2/prepare_v2_validation.py


In [16]:
%%writefile /content/dacon236749_v2/AST_LICENSE.txt
BSD 3-Clause License

Copyright (c) 2021, Yuan Gong
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

Writing /content/dacon236749_v2/AST_LICENSE.txt


In [17]:
INPUT_ARCHIVES = [{'sha256': '54c227b873cc3c424579fa06e8a06c442b91663de5b740b182f7cac30a3fac75', 'bytes': 346433653, 'model_revision': 'f826b80d28226b62986cc218e5cec390b1096902', 'name': 'v2_ast_assets.zip', 'parts': [{'name': 'v2_ast_assets.zip.part00', 'bytes': 83886080, 'sha256': 'f4f095b3c6062b341f6f2554c210254170da87281ee1eb547b4b6a09940764dc'}, {'name': 'v2_ast_assets.zip.part01', 'bytes': 83886080, 'sha256': 'd277ef91dcef6fe298c48ce23bbb3289b7b49e3748bd5fae2cf614f91c0c2e4d'}, {'name': 'v2_ast_assets.zip.part02', 'bytes': 83886080, 'sha256': 'df38da76de6f510582055ba33225502e7b9fa1747f487742f3dd762b12e629fb'}, {'name': 'v2_ast_assets.zip.part03', 'bytes': 83886080, 'sha256': 'b575be13671d5c3ffbc0ba74cf8a4c701c91d9270c0bffd473a7a105a623f96a'}, {'name': 'v2_ast_assets.zip.part04', 'bytes': 10889333, 'sha256': 'f2ade7b797f0ad384c0632b3d0388d4df01a9e70db6465325ac4bf9e8c2d51c3'}]}, {'name': 'v2_validation.zip', 'sha256': 'f08eef8f309fab3a176a89fe34fffeaf687fc99ffe4f16c60157addaf9796b9e', 'bytes': 217713960, 'parts': [{'name': 'v2_validation.zip.part00', 'sha256': '71ad85610171ddbab1f37bd6a20f13c6adfa5886b44d408a8983f645520876b0', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part00'}, {'name': 'v2_validation.zip.part01', 'sha256': 'a357bb091f4e5e4302e1c5b4ce33492850d25714d2c6d66593b80a5ca1e97a04', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part01'}, {'name': 'v2_validation.zip.part02', 'sha256': '899c0f4a3352da01d642417711ca8575c7328e1b6ec7072737edc0415a942444', 'bytes': 49941800, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part02'}]}]

## 3. 설치·데이터 검증·A100 비교·저장
학습에는 개발 manifest만 전달합니다. 선택과 가중치 해시를 확정한 다음 새 평가 집합을 엽니다. 각 epoch, 예측, 선택 근거, 제출 ZIP 재실행 결과는 Drive 실행 폴더에 남습니다.

In [18]:
import os
try:
    (RUN / 'code').mkdir(exist_ok=True)
    for p in WORK.glob('*.py'):
        shutil.copy2(p, RUN / 'code' / p.name)
    shutil.copy2(WORK / 'AST_LICENSE.txt', RUN / 'code' / 'AST_LICENSE.txt')
    (RUN / 'input_archives.json').write_text(json.dumps(INPUT_ARCHIVES, indent=2))
    packages = ['demucs==4.0.1','panns-inference==0.1.1','transformers==4.57.6',
                'librosa==0.10.2.post1','diffq==0.2.4','einops==0.8.1',
                'scikit-learn==1.8.0','soundfile==0.12.1','safetensors']
    with (RUN / 'install.log').open('w') as log:
        subprocess.run([sys.executable,'-m','pip','install','-q',*packages],
                       stdout=log,stderr=subprocess.STDOUT,check=True,timeout=600)
    print('INSTALL_FINISHED', flush=True)
    def sha256(path):
        with path.open('rb') as source:
            return hashlib.file_digest(source,'sha256').hexdigest()
    for archive in INPUT_ARCHIVES:
        destination = WORK / archive['name']
        with destination.open('wb') as output:
            for part in archive['parts']:
                source = DRIVE / part['name']
                assert sha256(source) == part['sha256'], f"Drive part differs: {source}"
                with source.open('rb') as incoming:
                    shutil.copyfileobj(incoming,output,8*1024**2)
        assert sha256(destination) == archive['sha256'], f"Archive differs: {destination}"
        with zipfile.ZipFile(destination) as packed:
            packed.extractall(WORK)
        print('VERIFIED_ARCHIVE', destination.name, flush=True)
    shutil.copy2(DRIVE / 'open.zip', WORK / 'open.zip')
    with zipfile.ZipFile(WORK / 'open.zip') as packed:
        packed.extractall(WORK / 'official')
    original = next((WORK / 'official').rglob('baseline_submit.zip'))
    assert sha256(original) == 'cc86f143915c95fd958a0066cbf14b0db9f4c41fd213310c3650b9fe870d0c70'
    BASELINE = WORK / 'baseline'
    with zipfile.ZipFile(original) as packed:
        packed.extractall(BASELINE)
    public = next((WORK / 'official').rglob('sample_submission.csv')).parent
    shutil.copytree(public,BASELINE / 'data',dirs_exist_ok=True)
    os.chdir(WORK)
    cmd = [sys.executable,'-u','run_v2.py','--assets',str(WORK / 'ast'),
           '--data',str(WORK / 'v2_validation'),'--baseline',str(BASELINE),
           '--original-zip',str(original),'--output',str(RUN),'--work',str(WORK),
           '--deadline',str(HARD_DEADLINE)]
    with (RUN / 'experiment.log').open('w',buffering=1) as log:
        process = subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT)
        while process.poll() is None:
            if time.time() >= HARD_DEADLINE - 60:
                process.terminate()
                process.wait(timeout=30)
                raise TimeoutError('Stopped at allocation limit')
            time.sleep(15)
            lines = (RUN / 'experiment.log').read_text().splitlines()
            print(lines[-1][:1000] if lines else 'Loading models...', flush=True)
        if process.returncode:
            print((RUN / 'experiment.log').read_text()[-16000:])
            raise RuntimeError(f'Experiment exited with {process.returncode}')
    from IPython.display import display
    display(json.loads((RUN / 'comparison.json').read_text()))
    display(json.loads((RUN / 'packaged_verification.json').read_text()))
    print('V2_SAVED',RUN,'Elapsed seconds',round(time.time()-SESSION_START),flush=True)
except Exception:  # noqa: BROAD_EXCEPT_OK
    (RUN / 'failure.txt').write_text(traceback.format_exc())
    print(traceback.format_exc(),flush=True)
    raise
finally:
    (RUN / 'release_requested.json').write_text(json.dumps({'unix':time.time(),
        'elapsed_seconds':time.time()-SESSION_START}))
    watchdog.cancel()
    runtime.unassign()

INSTALL_FINISHED
VERIFIED_ARCHIVE v2_ast_assets.zip
VERIFIED_ARCHIVE v2_validation.zip
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Checkpoint path: /content/dacon236749_v2/baseline/model/panns/Cnn14_mAP=0.431.pth
GPU number: 1
FEATURE 21/122 20f387562072c8d84584
FEATURE 42/122 43cae1f2da315de45add
FEATURE 61/122 78eedcae879fec19daff
FEATURE 81/122 a1a3e99628907df7c501
FEATURE 103/122 e7cb4dbc721887bf1d86
FEATURE 122/122 music_ef4d9f91f6bbf0d7
FROZEN_CACHE 450 embeddings
TRAIN_EPOCH AST_FROZEN 2/8 loss=0.345000 proxy=0.746352 head_grad=3.530240 backbone_grad=0.000000
TRAIN_EPOCH AST_FROZEN 7/8 loss=0.207654 proxy=0.809294 head_grad=2.501133 backbone_grad=0.000000
TRAIN_EPOCH AST_LAST_BLOCK 1/6 loss=0.182121 proxy=0.837507 head_grad=2.212255 backbone_grad=2.300445
TRAIN_EPOCH AST_LAST_BLOCK 3/6 loss=0.140132 proxy=0.852850 head_grad=2.297719 backbone_grad=3.670333
TRAIN_EPOCH AST_LAST_BLO

RuntimeError: Experiment exited with 1